# Amazon ML Challenge — Business Entity Resolution (Production)

Self-contained Kaggle notebook. Runs the validated Phase 1 blocking /
indexing / feature architecture (`normalize.py`, `blocking.py`,
`indexer.py`, `candidates.py`, `features.py`, `labels.py`,
`checkpoint.py`) against the **full** `train_source1/2/3.tsv` +
`train_ground_truth.tsv` and `test_source1/2/3.tsv` datasets.

Everything is checkpointed under `/kaggle/working/amazon_ml_solution/`
so an interrupted session can be resumed by re-running the same cell.

`train_10k` is available only as an optional smoke-test subset
(`SMOKE_TEST = True` below) — it is never used as the production
dataset.


## CELL 1 — Environment + configuration

In [6]:
import os
import sys
import json
import time
import random
import csv
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# SMOKE TEST
# ------------------------------------------------------------
# True  = small test run to verify the pipeline works
# False = full production run on the complete dataset
#
# START WITH True.
# ------------------------------------------------------------

SMOKE_TEST = True

SMOKE_TEST_S1_LIMIT = 2000
SMOKE_TEST_INDEX_LIMIT = 100_000


# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

USE_GPU = True


# ------------------------------------------------------------
# INDEX BACKEND
# ------------------------------------------------------------
# Smoke test -> memory
# Full run   -> sqlite
# ------------------------------------------------------------

INDEX_BACKEND = "memory" if SMOKE_TEST else "sqlite"


# ------------------------------------------------------------
# SHARDING
# ------------------------------------------------------------

CANDIDATE_SHARD_SIZE = (
    5_000 if SMOKE_TEST else 20_000
)

# Number of candidate/feature rows processed by the
# model in one prediction batch.
SCORE_BATCH_SIZE = 200_000


# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------

RANDOM_SEED = 42
random.seed(RANDOM_SEED)


# ============================================================
# WORKING DIRECTORIES
# ============================================================

# IMPORTANT:
# Never write anything into /kaggle/input.
# Kaggle input data is read-only.
# All generated files go into /kaggle/working.

BASE_DIR = Path("/kaggle/working/amazon_ml_solution")

SRC_DIR = BASE_DIR / "src"
IDX_DIR = BASE_DIR / "indexes"
CKPT_DIR = BASE_DIR / "checkpoints"
SHARD_DIR = CKPT_DIR / "shards"
OUT_DIR = BASE_DIR / "output"


# Create directories

for d in (
    BASE_DIR,
    SRC_DIR,
    IDX_DIR,
    CKPT_DIR,
    SHARD_DIR,
    OUT_DIR,
):
    d.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# PYTHON PATH
# ============================================================

sys.path.insert(
    0,
    str(SRC_DIR)
)


# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("=" * 70)
print("AMAZON ML ENTITY RESOLUTION")
print("KAGGLE CONFIGURATION")
print("=" * 70)

print(
    "BASE_DIR              :",
    BASE_DIR
)

print(
    "SMOKE_TEST            :",
    SMOKE_TEST
)

print(
    "SMOKE_TEST_S1_LIMIT   :",
    SMOKE_TEST_S1_LIMIT
)

print(
    "SMOKE_TEST_INDEX_LIMIT:",
    SMOKE_TEST_INDEX_LIMIT
)

print(
    "INDEX_BACKEND         :",
    INDEX_BACKEND
)

print(
    "USE_GPU               :",
    USE_GPU
)

print(
    "CANDIDATE_SHARD_SIZE  :",
    CANDIDATE_SHARD_SIZE
)

print(
    "SCORE_BATCH_SIZE      :",
    SCORE_BATCH_SIZE
)

print(
    "RANDOM_SEED           :",
    RANDOM_SEED
)

print("=" * 70)

AMAZON ML ENTITY RESOLUTION
KAGGLE CONFIGURATION
BASE_DIR              : /kaggle/working/amazon_ml_solution
SMOKE_TEST            : True
SMOKE_TEST_S1_LIMIT   : 2000
SMOKE_TEST_INDEX_LIMIT: 100000
INDEX_BACKEND         : memory
USE_GPU               : True
CANDIDATE_SHARD_SIZE  : 5000
SCORE_BATCH_SIZE      : 200000
RANDOM_SEED           : 42


## CELL 2 — Install dependencies

In [7]:
import sys

# ============================================================
# OPTIONAL DEPENDENCIES
# ============================================================

# ------------------------------------------------------------
# RapidFuzz
# ------------------------------------------------------------
# RapidFuzz is only a speed optimization.
# features.py has a pure-Python fallback using difflib.
# We do NOT try to install anything from the internet.
# ------------------------------------------------------------

try:
    import rapidfuzz  # noqa: F401
    HAVE_RAPIDFUZZ = True
    print("[setup] rapidfuzz available.")
except ImportError:
    HAVE_RAPIDFUZZ = False
    print("[setup] rapidfuzz not available.")
    print("[setup] Using the built-in difflib fallback.")


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------
# LightGBM is optional.
# If it is already installed, we can use it.
# If not, we use sklearn's
# HistGradientBoostingClassifier.
#
# IMPORTANT:
# No pip install is attempted because this Kaggle
# session currently has no internet access.
# ------------------------------------------------------------

HAVE_LIGHTGBM = False

try:
    import lightgbm

    HAVE_LIGHTGBM = True

    print("[setup] lightgbm available.")

except ImportError:

    HAVE_LIGHTGBM = False

    print("[setup] lightgbm not available.")
    print(
        "[setup] Will use "
        "HistGradientBoostingClassifier instead."
    )


# ============================================================
# CORE LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import sklearn

print(
    "[setup] numpy",
    np.__version__,
    "pandas",
    pd.__version__,
    "sklearn",
    sklearn.__version__
)

print(
    "[setup] HAVE_RAPIDFUZZ:",
    HAVE_RAPIDFUZZ
)

print(
    "[setup] HAVE_LIGHTGBM:",
    HAVE_LIGHTGBM
)

print("=" * 70)
print("[setup] Dependency check complete.")
print("=" * 70)

[setup] rapidfuzz not available.
[setup] Using the built-in difflib fallback.
[setup] lightgbm available.
[setup] numpy 2.0.2 pandas 2.3.3 sklearn 1.6.1
[setup] HAVE_RAPIDFUZZ: False
[setup] HAVE_LIGHTGBM: True
[setup] Dependency check complete.


## CELL 3 — Locate Kaggle datasets

In [8]:
import os
from pathlib import Path

def discover_root(required_files, search_root="/kaggle/input"):
    """Walk search_root for a directory containing every file in required_files."""
    root = Path(search_root)
    if not root.exists():
        return None
    for dirpath, _dirnames, filenames in os.walk(root):
        names = set(filenames)
        if all(f in names for f in required_files):
            return dirpath
    return None

TRAIN_FILES = ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv", "train_ground_truth.tsv"]
TEST_FILES = ["test_source1.tsv", "test_source2.tsv", "test_source3.tsv"]
TRAIN_10K_FILES = ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv", "train_ground_truth.tsv"]

TRAIN_ROOT = discover_root(TRAIN_FILES)
TEST_ROOT = discover_root(TEST_FILES)

if TRAIN_ROOT is None:
    raise FileNotFoundError(
        "Could not find a directory under /kaggle/input containing all of: "
        + ", ".join(TRAIN_FILES) + ". Attach the competition dataset to this kernel."
    )
if TEST_ROOT is None:
    raise FileNotFoundError(
        "Could not find a directory under /kaggle/input containing all of: "
        + ", ".join(TEST_FILES) + ". Attach the competition dataset to this kernel."
    )

S1_TRAIN = os.path.join(TRAIN_ROOT, "train_source1.tsv")
S2_TRAIN = os.path.join(TRAIN_ROOT, "train_source2.tsv")
S3_TRAIN = os.path.join(TRAIN_ROOT, "train_source3.tsv")
GT_TRAIN = os.path.join(TRAIN_ROOT, "train_ground_truth.tsv")

S1_TEST = os.path.join(TEST_ROOT, "test_source1.tsv")
S2_TEST = os.path.join(TEST_ROOT, "test_source2.tsv")
S3_TEST = os.path.join(TEST_ROOT, "test_source3.tsv")

print("TRAIN_ROOT:", TRAIN_ROOT)
print("  S1_TRAIN:", S1_TRAIN)
print("  S2_TRAIN:", S2_TRAIN)
print("  S3_TRAIN:", S3_TRAIN)
print("  GT_TRAIN:", GT_TRAIN)
print("TEST_ROOT :", TEST_ROOT)
print("  S1_TEST :", S1_TEST)
print("  S2_TEST :", S2_TEST)
print("  S3_TEST :", S3_TEST)

TRAIN_ROOT: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train
  S1_TRAIN: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv
  S2_TRAIN: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
  S3_TRAIN: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv
  GT_TRAIN: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_ground_truth.tsv
TEST_ROOT : /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/test
  S1_TEST : /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/test/test_source1.tsv
  S2_TEST : /kaggle/input/datasets/tejasweerajp

## CELL 4 — Validate files/columns

In [9]:
import csv

EXPECTED_SOURCE_COLUMNS = {"entity_id", "business_name", "business_address", "country"}
EXPECTED_GT_COLUMNS = {"source1_entity_id", "matched_entity_ids"}

def read_header(path):
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        return next(reader)

def validate_columns(path, expected, label):
    header = read_header(path)
    header_set = set(header)
    missing = expected - header_set
    status = "OK" if not missing else "MISSING " + str(missing)
    print(f"[validate] {label:10s} {path}")
    print(f"           columns: {header}")
    print(f"           status : {status}")
    if missing:
        raise ValueError(f"{label} at {path} is missing expected columns: {missing}")
    return header

validate_columns(S1_TRAIN, EXPECTED_SOURCE_COLUMNS, "S1_TRAIN")
validate_columns(S2_TRAIN, EXPECTED_SOURCE_COLUMNS, "S2_TRAIN")
validate_columns(S3_TRAIN, EXPECTED_SOURCE_COLUMNS, "S3_TRAIN")
validate_columns(GT_TRAIN, EXPECTED_GT_COLUMNS, "GT_TRAIN")
validate_columns(S1_TEST, EXPECTED_SOURCE_COLUMNS, "S1_TEST")
validate_columns(S2_TEST, EXPECTED_SOURCE_COLUMNS, "S2_TEST")
validate_columns(S3_TEST, EXPECTED_SOURCE_COLUMNS, "S3_TEST")

print("\nAll expected columns present.")

[validate] S1_TRAIN   /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv
           columns: ['entity_id', 'business_name', 'business_address', 'country']
           status : OK
[validate] S2_TRAIN   /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
           columns: ['entity_id', 'business_name', 'business_address', 'country']
           status : OK
[validate] S3_TRAIN   /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv
           columns: ['entity_id', 'business_name', 'business_address', 'country']
           status : OK
[validate] GT_TRAIN   /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_ground_truth.tsv
           columns: ['source1_entity_id', 'matched_entity_ids']
           s

## CELL 5 — Dataset statistics

In [10]:
def count_data_rows(path):
    """Row count excluding the header. Single sequential pass, O(file size)."""
    n = -1
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        for n, _ in enumerate(f):
            pass
    return n  # header counted as row -1, so this is (lines - 1) once n starts at -1

t0 = time.time()
counts = {
    "S1_TRAIN": count_data_rows(S1_TRAIN),
    "S2_TRAIN": count_data_rows(S2_TRAIN),
    "S3_TRAIN": count_data_rows(S3_TRAIN),
    "GT_TRAIN": count_data_rows(GT_TRAIN),
    "S1_TEST": count_data_rows(S1_TEST),
    "S2_TEST": count_data_rows(S2_TEST),
    "S3_TEST": count_data_rows(S3_TEST),
}
print(f"[stats] row counts (took {time.time()-t0:.1f}s):")
for k, v in counts.items():
    print(f"  {k:10s}: {v:,}")

N_S1_TRAIN = counts["S1_TRAIN"]
N_S2_TRAIN = counts["S2_TRAIN"]
N_S3_TRAIN = counts["S3_TRAIN"]
N_S1_TEST = counts["S1_TEST"]
N_S2_TEST = counts["S2_TEST"]
N_S3_TEST = counts["S3_TEST"]

with open(CKPT_DIR / "dataset_counts.json", "w") as f:
    json.dump(counts, f, indent=2)

[stats] row counts (took 131.2s):
  S1_TRAIN  : 2,206,821
  S2_TRAIN  : 5,034,616
  S3_TRAIN  : 5,285,603
  GT_TRAIN  : 2,206,821
  S1_TEST   : 1,732,544
  S2_TEST   : 4,887,273
  S3_TEST   : 5,082,316


## CELL 6 — Normalization utilities

Writes the validated Phase 1 normalization module (`normalize.py`) plus its small dependents (`similarity.py`, `data_loader.py`) to `SRC_DIR` and imports them. `normalize_business_name`/`normalize_address`/`normalize_text` are UNCHANGED from Phase 1 -- every blocking key and frequency cap downstream was tuned against their exact behavior. `normalize_unicode_preserving` / `normalize_business_name_script_agnostic` / `is_non_latin` are the Phase 1 fix that gives `features.py` real signal on non-Latin (Devanagari/Telugu/Kannada/Bengali, ...) business names instead of collapsing to `""`.

In [11]:
import pathlib
pathlib.Path(SRC_DIR).mkdir(parents=True, exist_ok=True)

pathlib.Path(SRC_DIR, "normalize.py").write_text(r'''"""
normalize.py
============================================================
PRESERVED: normalize_text / normalize_business_name /
normalize_address are UNCHANGED from the original implementation
and are still what blocking.py's ASCII-based keys (NP6, NP8, NS6,
FL, FL3, F2, FML, AN, A2, AW2, AL2, NL, A4W, AT2) and the
frequency-cap tuning in blocking.DEFAULT_FREQ_CAPS were built and
measured against. Do not change their behavior -- that would
silently invalidate every recall number in output/*.txt.

ADDED: normalize_unicode_preserving(), used by features.py (not
blocking.py, which already had its own equivalent logic inlined
in make_unicode_name_keys). This closes a real gap found during
the Phase 1 audit:

  normalize_business_name() / normalize_address() strip every
  character outside [a-z0-9\\s] AFTER NFKD decomposition. NFKD
  decomposition only helps scripts where the accent is a separate
  combining character on top of a Latin base letter (e.g. "e" +
  combining acute). It does nothing for Devanagari, Telugu,
  Kannada, Bengali, etc. -- those characters simply have no
  a-z0-9 equivalent, so they are deleted outright.

  Example measured in output/s2_missed_matches_inspection.tsv:
    "एसएस सिस्टम्स लिमिटेड"  -->  normalize_business_name()  -->  ""

  blocking.py's make_unicode_name_keys() already generates
  candidate-blocking keys straight from the untouched Unicode
  string, which is why these pairs DO show up as candidates.
  But features.py currently computes every name/address
  similarity feature from normalize_business_name() /
  normalize_address() alone. Once both sides normalize to "",
  every similarity feature for that pair becomes 0 (or the
  hard-coded "missing" default) regardless of whether the two
  Unicode strings are near-identical or completely unrelated.
  The ML model in Phase 2 would then have no signal at all to
  distinguish a true match from a false positive on this
  subset -- i.e. blocking recall is fine for non-Latin scripts,
  but downstream precision/recall on those rows would collapse.

  normalize_unicode_preserving() keeps letters/digits/combining
  marks in ANY script (via unicodedata category, not a Latin
  regex) so features.py can compute a second, script-agnostic
  set of similarity features and fall back to it whenever the
  ASCII-folded normalization is empty on either side.
"""

import re
import unicodedata


def normalize_text(text):
    """
    Basic normalization for business names and addresses.
    UNCHANGED -- do not modify, see module docstring.
    """

    if not text:
        return ""

    text = str(text)
    text = text.lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_business_name(name):
    """
    UNCHANGED -- do not modify, see module docstring.
    """

    name = normalize_text(name)

    suffixes = [
        "incorporated",
        "inc",
        "corporation",
        "corp",
        "limited",
        "ltd",
        "llc",
        "company",
        "co",
        "private limited",
        "pvt ltd",
    ]

    words = name.split()

    while words and " ".join(words[-2:]) in suffixes:
        words = words[:-2]

    while words and words[-1] in suffixes:
        words = words[:-1]

    return " ".join(words)


def normalize_address(address):
    """
    UNCHANGED -- do not modify, see module docstring.
    """

    address = normalize_text(address)

    replacements = {
        " street ": " st ",
        " road ": " rd ",
        " avenue ": " ave ",
        " boulevard ": " blvd ",
        " drive ": " dr ",
        " lane ": " ln ",
        " highway ": " hwy ",
        " apartment ": " apt ",
    }

    address = " " + address + " "

    for old, new in replacements.items():
        address = address.replace(old, new)

    return address.strip()


# ============================================================
# NEW: script-agnostic normalization for FEATURE calculation
# ============================================================

# A small set of business-suffix WORDS that are meaningful in
# transliterated/English form even inside an otherwise non-Latin
# string (e.g. "Pvt Ltd" tacked onto a Devanagari name). Matching
# the ASCII suffix list keeps this consistent with
# normalize_business_name() rather than inventing a second list.
_ASCII_SUFFIX_WORDS = {
    "incorporated", "inc", "corporation", "corp", "limited", "ltd",
    "llc", "company", "co", "private", "pvt",
}


def normalize_unicode_preserving(text):
    """
    Like normalize_text(), but keeps letters/digits/marks from
    ANY script instead of only [a-z0-9]. Used by features.py as a
    fallback whenever normalize_business_name()/normalize_address()
    collapse to "" for a non-Latin string, so similarity features
    still carry real signal for India-language business names.

    NOT used by blocking.py: blocking already has its own,
    independently-tuned Unicode key logic (make_unicode_name_keys)
    and changing that would move the measured recall numbers.
    """

    if not text:
        return ""

    text = str(text).strip()
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text).casefold()

    out_chars = []
    for ch in text:
        category = unicodedata.category(ch)
        if category.startswith("L") or category.startswith("N") or category.startswith("M"):
            out_chars.append(ch)
        else:
            out_chars.append(" ")

    text = re.sub(r"\s+", " ", "".join(out_chars)).strip()

    return text


def normalize_business_name_script_agnostic(name):
    """
    normalize_unicode_preserving() plus trailing-suffix stripping,
    for business names specifically. Suffix stripping only removes
    ASCII suffix words (see _ASCII_SUFFIX_WORDS); non-Latin legal
    suffixes are left in place since a hard-coded ASCII list can't
    identify them, and leaving them in does not hurt similarity
    scoring the way an empty string does.
    """

    text = normalize_unicode_preserving(name)
    if not text:
        return ""

    words = text.split()
    while words and words[-1] in _ASCII_SUFFIX_WORDS:
        words = words[:-1]

    return " ".join(words)


def is_non_latin(text):
    """
    True if the ASCII-folded normalization would lose real content
    -- i.e. normalize_business_name(text) would be "" or much
    shorter than normalize_unicode_preserving(text). Used by
    features.py to decide whether to trust the ASCII-folded
    features or fall back to the script-agnostic ones.
    """

    ascii_form = normalize_text(text)
    unicode_form = normalize_unicode_preserving(text)

    if not unicode_form:
        return False

    return len(ascii_form.replace(" ", "")) < 0.5 * len(unicode_form.replace(" ", ""))


if __name__ == "__main__":

    examples = [
        "Payne Enterprises",
        "एसएस सिस्टम्स लिमिटेड",
        "శివ మేనేజ్‌మెంట్ ప్రైవేట్ లిమిటెడ్",
        "ಬಾಂಬೆ ಪವರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್",
    ]

    for name in examples:
        print(f"{name!r}")
        print(f"  ascii-folded : {normalize_business_name(name)!r}")
        print(f"  script-agnostic: {normalize_business_name_script_agnostic(name)!r}")
        print(f"  is_non_latin : {is_non_latin(name)}")
''', encoding="utf-8")
pathlib.Path(SRC_DIR, "similarity.py").write_text(r'''from difflib import SequenceMatcher
import re

from normalize import (
    normalize_business_name,
    normalize_address
)


def text_similarity(text1, text2):
    """
    Character-level similarity.
    """

    if not text1 or not text2:
        return 0.0

    return SequenceMatcher(
        None,
        text1,
        text2
    ).ratio()


def token_similarity(text1, text2):
    """
    Compare sets of words.

    Example:

    "payne enterprises"
    "payne enterprises llc"

    share the important tokens.
    """

    if not text1 or not text2:
        return 0.0

    tokens1 = set(text1.split())
    tokens2 = set(text2.split())

    if not tokens1 or not tokens2:
        return 0.0

    intersection = tokens1 & tokens2
    union = tokens1 | tokens2

    return len(intersection) / len(union)


def name_similarity(name1, name2):
    """
    Combine character and token similarity.
    """

    name1 = normalize_business_name(name1)
    name2 = normalize_business_name(name2)

    char_score = text_similarity(
        name1,
        name2
    )

    token_score = token_similarity(
        name1,
        name2
    )

    # Give more importance to character similarity,
    # while also considering shared words.
    return (
        0.6 * char_score
        +
        0.4 * token_score
    )


def address_similarity(address1, address2):
    """
    Compare normalized addresses.
    """

    address1 = normalize_address(address1)
    address2 = normalize_address(address2)

    return text_similarity(
        address1,
        address2
    )


def country_match(country1, country2):
    """
    Compare countries.

    1.0 = same country
    0.0 = different country
    """

    if not country1 or not country2:
        return 0.0

    return 1.0 if (
        country1.strip().lower()
        ==
        country2.strip().lower()
    ) else 0.0


def extract_address_numbers(address):
    """
    Extract numeric parts from an address.
    """

    if not address:
        return []

    address = normalize_address(address)

    return re.findall(
        r"\d+",
        address
    )


def address_number_match(address1, address2):
    """
    Compare numeric parts of addresses.

    1.0 = numbers match
    0.0 = numbers differ
    0.5 = number unavailable in one/both
    """

    numbers1 = extract_address_numbers(
        address1
    )

    numbers2 = extract_address_numbers(
        address2
    )

    if not numbers1 or not numbers2:
        return 0.5

    if numbers1 == numbers2:
        return 1.0

    return 0.0


def calculate_similarity(row1, row2):
    """
    Calculate similarity features.
    """

    name_score = name_similarity(
        row1["business_name"],
        row2["business_name"]
    )

    address_score = address_similarity(
        row1["business_address"],
        row2["business_address"]
    )

    address_number_score = address_number_match(
        row1["business_address"],
        row2["business_address"]
    )

    country_score = country_match(
        row1["country"],
        row2["country"]
    )

    return {
        "name_similarity": name_score,
        "address_similarity": address_score,
        "address_number_match": address_number_score,
        "country_match": country_score
    }


if __name__ == "__main__":

    s1 = {
        "business_name": "Payne Enterprises",
        "business_address": "3315 Fremont Street, Peoria, IL",
        "country": "US"
    }

    s2 = {
        "business_name": "Payne Énterprises LLC",
        "business_address": "3315 FREMONT ST, PEORIA, IL",
        "country": "US"
    }

    false_match = {
        "business_name": "International Automation Consultants Corp",
        "business_address": "338 Rev Walton Dr, Lockport, Illinois",
        "country": "US"
    }

    true_match = {
        "business_name": "International Automation Consultants Inc",
        "business_address": "329 Rev Walton Drive, Lockport, IL",
        "country": "US"
    }

    print("=" * 70)
    print("PAYNE ENTERPRISES")
    print("=" * 70)

    print(
        calculate_similarity(
            s1,
            s2
        )
    )

    print("\n" + "=" * 70)
    print("INTERNATIONAL AUTOMATION")
    print("=" * 70)

    print(
        calculate_similarity(
            true_match,
            false_match
        )
    )''', encoding="utf-8")
pathlib.Path(SRC_DIR, "data_loader.py").write_text(r'''import csv


def read_tsv(path):
    """
    Read a TSV file one row at a time.
    """

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="replace",
        newline=""
    ) as f:

        reader = csv.DictReader(f, delimiter="\t")

        for row in reader:
            yield row


def preview_file(path, n=5):

    print("=" * 70)
    print("FILE:", path)
    print("=" * 70)

    for i, row in enumerate(read_tsv(path)):

        print(row)

        if i + 1 >= n:
            break


if __name__ == "__main__":

    preview_file(
        r"dataset\train\train_source1.tsv",
        n=5
    )''', encoding="utf-8")

print("Wrote:", ['normalize.py', 'similarity.py', 'data_loader.py'])

Wrote: ['normalize.py', 'similarity.py', 'data_loader.py']


In [12]:
from normalize import (
    normalize_text, normalize_business_name, normalize_address,
    normalize_unicode_preserving, normalize_business_name_script_agnostic, is_non_latin,
)
from data_loader import read_tsv

# Quick self-test: confirms the non-Latin gap is actually fixed.
example = "\u090f\u0938\u090f\u0938 \u0938\u093f\u0938\u094d\u091f\u092e\u094d\u0938 \u0932\u093f\u092e\u093f\u091f\u0947\u0921"
print("raw                :", example)
print("ascii-folded       :", repr(normalize_business_name(example)))
print("script-agnostic    :", repr(normalize_business_name_script_agnostic(example)))
print("is_non_latin       :", is_non_latin(example))
assert normalize_business_name(example) == "", "expected ASCII-folded form to be empty for pure Devanagari"
assert normalize_business_name_script_agnostic(example) != "", "script-agnostic fallback must not be empty"
print("\n[normalize] OK")

raw                : एसएस सिस्टम्स लिमिटेड
ascii-folded       : ''
script-agnostic    : 'एसएस सिस्टम्स लिमिटेड'
is_non_latin       : True

[normalize] OK


## CELL 7 — Blocking-key generation

Writes `blocking.py` unchanged from Phase 1: source-independent name/address/Unicode blocking-key generators plus `DEFAULT_FREQ_CAPS`, the per-key-type frequency ceiling that keeps buckets like `AL2`/`NS6` from exploding (see `PHASE1_ARCHITECTURE_AUDIT.md` sections 3-5). This is the multi-route, frequency-aware blocking design the brief asks for -- name signals, Unicode-aware name signals, compact-name signals, address signals, address-number signals, and country-aware scoping (via `compose_key` in `indexer.py`, applied next cell).

In [13]:
import pathlib
pathlib.Path(SRC_DIR).mkdir(parents=True, exist_ok=True)

pathlib.Path(SRC_DIR, "blocking.py").write_text(r'''"""
blocking.py
============================================================
Blocking-key generation for Business Entity Resolution.

This module is SOURCE-INDEPENDENT: the same functions are used
for Source 1, Source 2 and Source 3, and for both the 10K
development data and the full 5M+ record production data.

Frequency thresholds are NOT computed here -- they are computed
by indexer.py against whatever data was actually indexed.

Key types produced:

Name keys:
    NP6, NP8, NS6, FL, FL3, F2, FML
    UNP4, UNP6, UFL

Address keys:
    AN, A2, AW2, AL2, NL
    A4W, AT2
    RAT20 = Rare Address Token, frequency capped at 20
"""


import re
import unicodedata

from normalize import normalize_business_name, normalize_address


# ============================================================
# UNICODE NAME BLOCKING
# ============================================================

def make_unicode_name_keys(name):
    """
    Unicode-preserving name keys.

    Keeps Hindi, Kannada, Gujarati, etc. instead of deleting
    non-ASCII characters.
    """

    if not name:
        return []

    n = unicodedata.normalize("NFKC", str(name)).casefold()
    n = " ".join(n.split())

    if not n:
        return []

    keys = []

    compact = "".join(ch for ch in n if not ch.isspace())

    if len(compact) >= 4:
        keys.append(("UNP4", compact[:4]))

    if len(compact) >= 6:
        keys.append(("UNP6", compact[:6]))

    words = n.split()

    if len(words) >= 2:
        first = words[0]
        last = words[-1]

        if len(first) >= 2 and len(last) >= 2:
            keys.append(("UFL", first[:3] + "|" + last[:3]))

    return keys


# ============================================================
# NAME BLOCKING
# ============================================================

def make_name_keys(name):
    """
    Generate selective name-blocking keys.

    Generic single-word keys are avoided because they create
    enormous candidate buckets.
    """

    n = normalize_business_name(name)

    if not n:
        return []

    keys = []

    compact = n.replace(" ", "")

    if len(compact) >= 6:
        keys.append(("NP6", compact[:6]))

    if len(compact) >= 8:
        keys.append(("NP8", compact[:8]))

    if len(compact) >= 6:
        keys.append(("NS6", compact[-6:]))

    words = n.split()

    if len(words) >= 2:
        first, last = words[0], words[-1]

        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FL", first[:4] + "|" + last[:4]))
            keys.append(("FL3", first[:3] + "|" + last[:5]))

        w1, w2 = words[0], words[1]

        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("F2", w1[:5] + "|" + w2[:5]))

    if len(words) >= 3:
        first, middle, last = words[0], words[-2], words[-1]

        if len(first) >= 3 and len(last) >= 3:
            keys.append(
                ("FML", first[:3] + "|" + middle[:3] + "|" + last[:3])
            )

    return keys


# ============================================================
# ADDRESS BLOCKING
# ============================================================

def make_address_keys(address):
    """
    Generate selective address-blocking keys.

    We deliberately do NOT use a bare number as a key.
    """

    a = normalize_address(address)

    if not a:
        return []

    keys = []

    words = a.split()

    if not words:
        return []

    numbers = [w for w in words if w.isdigit()]
    non_numeric = [w for w in words if not w.isdigit()]

    if numbers:
        number = numbers[0]

        for word in words:
            if not word.isdigit() and len(word) >= 3:
                keys.append(("AN", number + "|" + word[:6]))
                break

        if len(non_numeric) >= 2:
            keys.append(
                (
                    "A2",
                    number
                    + "|"
                    + non_numeric[0][:4]
                    + "|"
                    + non_numeric[1][:4],
                )
            )

    if len(non_numeric) >= 2:
        w1, w2 = non_numeric[0], non_numeric[1]

        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AW2", w1[:5] + "|" + w2[:5]))

        w1, w2 = non_numeric[-2], non_numeric[-1]

        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AL2", w1[:5] + "|" + w2[:5]))

    if numbers and non_numeric:
        number = numbers[0]
        last_word = non_numeric[-1]

        if len(last_word) >= 3:
            keys.append(("NL", number + "|" + last_word[:6]))

    return keys


# ============================================================
# RARE ADDRESS TOKEN BLOCKING
# ============================================================

# Very common address tokens are weak blocking signals.
# They are excluded completely.
GENERIC_ADDRESS_WORDS = {
    "street", "st", "road", "rd", "avenue", "ave",
    "lane", "ln", "drive", "dr", "boulevard", "blvd",
    "highway", "hwy", "near", "opposite", "opp",
    "building", "bldg", "floor", "fl", "block",
    "district", "city", "state", "india",
    "private", "limited", "ltd", "pvt",
}


def make_rare_address_token_keys(address):
    """
    Generate individual informative address-token keys.

    These keys are intentionally broad at generation time.
    indexer.py applies the frequency cap against the actual
    dataset being indexed.

    RAT20 therefore means:

        Rare Address Token
        maximum bucket frequency = 20

    This is useful when the business name is corrupted or written
    in another script but some distinctive address token survives.
    """

    a = normalize_address(address)

    if not a:
        return []

    tokens = a.split()

    if not tokens:
        return []

    keys = []
    seen = set()

    for tok in tokens:

        # Numbers alone are not distinctive enough.
        if tok.isdigit():
            continue

        # Ignore extremely short tokens.
        if len(tok) < 4:
            continue

        # Ignore generic address words.
        if tok in GENERIC_ADDRESS_WORDS:
            continue

        # Normalize token representation.
        token = tok[:12]

        if not token:
            continue

        if token in seen:
            continue

        seen.add(token)

        keys.append(("RAT20", token))

    return keys


# ============================================================
# AT2 BLOCKING
# ============================================================

def make_at2_keys(address):
    """
    Generate informative address-token-pair keys.

    Unlike AW2/AL2, this does not depend on token position.
    """

    a = normalize_address(address)

    if not a:
        return []

    tokens = a.split()

    informative = []

    for tok in tokens:

        if tok.isdigit():
            continue

        if len(tok) < 4:
            continue

        if tok in GENERIC_ADDRESS_WORDS:
            continue

        informative.append(tok)

    if len(informative) < 2:
        return []

    # Keep at most 6 informative tokens.
    informative = sorted(
        informative,
        key=lambda x: (-len(x), x)
    )[:6]

    keys = []
    seen = set()

    for i in range(len(informative) - 1):

        w1 = informative[i][:8]
        w2 = informative[i + 1][:8]

        if w1 == w2:
            continue

        a1, a2 = sorted((w1, w2))

        key_value = a1 + "|" + a2

        if key_value not in seen:
            seen.add(key_value)
            keys.append(("AT2", key_value))

    return keys


# ============================================================
# A4 BLOCKING
# ============================================================

A4_WINDOW = 3


def make_a4_keys(address, window=A4_WINDOW):
    """
    Numeric address component + nearby normalized address words.

    Windowed implementation keeps key generation O(n).
    """

    a = normalize_address(address)

    if not a:
        return []

    tokens = a.split()

    number_positions = [
        i for i, t in enumerate(tokens)
        if t.isdigit()
    ]

    if not number_positions:
        return []

    keys = []
    seen = set()

    for pos in number_positions:

        number = tokens[pos]

        nearby = []

        for offset in range(1, window + 1):

            for idx in (pos - offset, pos + offset):

                if 0 <= idx < len(tokens):

                    tok = tokens[idx]

                    if not tok.isdigit() and len(tok) >= 2:
                        nearby.append(tok)

        nearby = list(dict.fromkeys(nearby))[:3]

        for i in range(len(nearby)):

            for j in range(i + 1, len(nearby)):

                w1, w2 = sorted(
                    (nearby[i][:6], nearby[j][:6])
                )

                key_value = (
                    number
                    + "|"
                    + w1
                    + "|"
                    + w2
                )

                if key_value not in seen:
                    seen.add(key_value)
                    keys.append(("A4W", key_value))

    return keys


# ============================================================
# COMBINED
# ============================================================

def make_all_keys(name, address):
    """
    Return every blocking key for a record.
    """

    return (
        make_name_keys(name)
        + make_unicode_name_keys(name)
        + make_address_keys(address)
        + make_rare_address_token_keys(address)
        + make_a4_keys(address)
        + make_at2_keys(address)
    )


# ============================================================
# KEY TYPE REGISTRY
# ============================================================

NAME_KEY_TYPES = {
    "NP6",
    "NP8",
    "NS6",
    "FL",
    "FL3",
    "F2",
    "FML",
    "UNP4",
    "UNP6",
    "UFL",
}


ADDRESS_KEY_TYPES = {
    "AN",
    "A2",
    "AW2",
    "AL2",
    "NL",
    "A4W",
    "AT2",
    "RAT20",
}


ALL_KEY_TYPES = NAME_KEY_TYPES | ADDRESS_KEY_TYPES


# ============================================================
# DEFAULT FREQUENCY CAPS
# ============================================================

DEFAULT_FREQ_CAPS = {
    # Name blockers
    "NP6": 300,
    "NP8": 300,
    "NS6": 300,
    "FL": 400,
    "FL3": 400,
    "F2": 400,
    "FML": 400,

    # Address blockers
    "AN": 400,
    "A2": 400,
    "AW2": 400,

    # AL2 was previously the major explosion source,
    # so keep its tighter cap.
    "AL2": 80,

    "NL": 400,

    # Specialized blockers
    "A4W": 10,
    "AT2": 20,

    # NEW:
    # Rare address token blocker.
    #
    # Only address tokens occurring <=20 times in the
    # indexed source are allowed to generate candidates.
    "RAT20": 20,

    # Unicode name blockers
    "UNP4": 300,
    "UNP6": 300,
    "UFL": 300,
}''', encoding="utf-8")

print("Wrote:", ['blocking.py'])

Wrote: ['blocking.py']


In [14]:
from blocking import (
    make_name_keys, make_unicode_name_keys, make_address_keys,
    make_a4_keys, make_at2_keys, make_rare_address_token_keys,
    make_all_keys, DEFAULT_FREQ_CAPS, NAME_KEY_TYPES, ADDRESS_KEY_TYPES,
)

print("[blocking] key types:", sorted(NAME_KEY_TYPES | ADDRESS_KEY_TYPES))
print("[blocking] DEFAULT_FREQ_CAPS:")
for k, v in sorted(DEFAULT_FREQ_CAPS.items()):
    print(f"    {k:6s}: {v}")

_example = next(read_tsv(S1_TRAIN))
print("\n[blocking] example S1 row:", _example)
print("[blocking] keys for it   :", make_all_keys(_example.get("business_name", ""), _example.get("business_address", "")))

[blocking] key types: ['A2', 'A4W', 'AL2', 'AN', 'AT2', 'AW2', 'F2', 'FL', 'FL3', 'FML', 'NL', 'NP6', 'NP8', 'NS6', 'RAT20', 'UFL', 'UNP4', 'UNP6']
[blocking] DEFAULT_FREQ_CAPS:
    A2    : 400
    A4W   : 10
    AL2   : 80
    AN    : 400
    AT2   : 20
    AW2   : 400
    F2    : 400
    FL    : 400
    FL3   : 400
    FML   : 400
    NL    : 400
    NP6   : 300
    NP8   : 300
    NS6   : 300
    RAT20 : 20
    UFL   : 300
    UNP4  : 300
    UNP6  : 300

[blocking] example S1 row: {'entity_id': 'S1-925783039', 'business_name': "Orelee's Barbershop", 'business_address': '1795 Westchester Drive, High Point, NC', 'country': 'US'}
[blocking] keys for it   : [('NP6', 'orelee'), ('NP8', 'oreleesb'), ('NS6', 'ershop'), ('FL', 'orel|barb'), ('FL3', 'ore|barbe'), ('FML', 'ore|s|bar'), ('UNP4', 'orel'), ('UNP6', 'orelee'), ('UFL', 'ore|bar'), ('AN', '1795|westch'), ('A2', '1795|west|dr'), ('RAT20', 'westchester'), ('RAT20', 'high'), ('RAT20', 'point'), ('A4W', '1795|dr|westch'), ('A4W', '179

## CELL 8 — Build S2 index

Writes `indexer.py` (compact postings index, capped `bucket(..., limit=...)`, periodic-commit + manifest-based resume, `discover_dataset_root`) and `checkpoint.py` (generic shard/manifest utilities reused by later stages), then builds the **full** S2 TRAIN index. SQLite backend by default -- bounded memory at 5M+ records. If this cell is interrupted, re-running it resumes from the last committed flush instead of rebuilding from row 0 (see `indexer.py`'s `SqliteIndex`/`build_index` docstring).

In [15]:
import pathlib
pathlib.Path(SRC_DIR).mkdir(parents=True, exist_ok=True)

pathlib.Path(SRC_DIR, "indexer.py").write_text(r'''"""
indexer.py
============================================================
CHANGES from the original (see PHASE1_ARCHITECTURE_AUDIT.md,
"Existing SQLite architecture" and "Checkpoint/resume plan"):

1. bucket() now accepts an optional `limit`. candidates.py passes
   limit = cap + 1. Previously SqliteIndex.bucket() always ran
   `SELECT suffix FROM postings WHERE key_type=? AND key_value=?`
   with no LIMIT and called fetchall() -- for a hot key (a common
   6-character name prefix, say) whose bucket is far larger than
   its frequency cap, this fetched and materialized the ENTIRE
   oversized bucket just to immediately discard it in
   candidates.py's `if bucket_size > cap: continue`. At full
   production scale (S2/S3 ~5M rows each, vs. the 100K-row index
   the 98.45%/97.96% numbers were measured against) hot buckets
   are proportionally larger, so this was on track to become one
   of the biggest avoidable costs in candidate generation. With a
   LIMIT of cap+1, an oversized bucket now costs O(cap) to detect
   as oversized, not O(bucket size).

2. SqliteIndex now commits periodically (every flush(), not only
   at finalize()) and writes a small JSON manifest recording how
   many source rows have been indexed. On restart, build_index()
   checks the manifest: if a source's index previously finished,
   it is reused as-is (no rebuild); if it was left partway
   through, indexing resumes from the last completed row instead
   of starting over. This is what makes a Kaggle session
   interruption DURING index-building recoverable -- previously
   the whole build sat inside one open SQLite transaction with no
   recorded progress, so a kill at row 4,900,000 of 5,000,000 lost
   the entire build.

3. Kaggle-path discovery: `discover_dataset_root()` walks
   /kaggle/input for a directory containing a train_source1.tsv,
   instead of a hard-coded local path. Falls back to a path the
   caller supplies, so this remains usable outside Kaggle too.

Everything else (MemoryIndex, split_entity_id, join_entity_id,
compose_key, the key-generation call sites) is UNCHANGED --
these are what the measured recall numbers depend on.
"""

import json
import os
import sqlite3
from collections import defaultdict
from pathlib import Path

from blocking import (
    make_name_keys,
    make_unicode_name_keys,
    make_address_keys,
    make_a4_keys,
    make_at2_keys,
)
from data_loader import read_tsv


def split_entity_id(entity_id):
    if "-" not in entity_id:
        return "", entity_id
    prefix, suffix = entity_id.split("-", 1)
    if suffix.isdigit():
        return prefix, int(suffix)
    return prefix, suffix


def join_entity_id(prefix, suffix):
    return f"{prefix}-{suffix}"


def compose_key(country, key_value):
    return (country or "").strip().upper() + "||" + key_value


# ============================================================
# KAGGLE PATH DISCOVERY
# ============================================================

def discover_dataset_root(required_files=("train_source1.tsv",), search_root="/kaggle/input"):
    """
    Walk search_root looking for a directory that contains every
    file in required_files. Returns the first match, or None.

    Avoids hard-coding a specific Kaggle dataset slug (the
    notebook skeleton in notebookc647d5dee5.ipynb hard-codes
    ".../tejasweerajput/amazon-ml-challenge-2026-student-resource/
    student_resource" -- if the dataset is re-attached under a
    different slug, or a new version, or the person forks the
    notebook, every path breaks silently). This walks the tree
    once at startup instead.
    """
    root = Path(search_root)
    if not root.exists():
        return None

    for dirpath, _dirnames, filenames in os.walk(root):
        filenames = set(filenames)
        if all(f in filenames for f in required_files):
            return dirpath

    return None


# ============================================================
# IN-MEMORY BACKEND (unchanged)
# ============================================================

class MemoryIndex:

    def __init__(self, source_prefix):
        self.source_prefix = source_prefix
        self.postings = defaultdict(lambda: defaultdict(list))
        self.attrs = {}
        self.count = 0

    def add_record(self, entity_id, name, address, country):

        _, suffix = split_entity_id(entity_id)
        self.attrs[suffix] = (name, address, country)

        for key_type, key_value in make_name_keys(name) + make_unicode_name_keys(name):
            self.postings[key_type][compose_key(country, key_value)].append(suffix)

        for key_type, key_value in make_address_keys(address):
            self.postings[key_type][compose_key(country, key_value)].append(suffix)

        for key_type, key_value in make_a4_keys(address) + make_at2_keys(address):
            self.postings[key_type][compose_key(country, key_value)].append(suffix)

        self.count += 1

    def bucket(self, key_type, key_value, limit=None):
        result = self.postings.get(key_type, {}).get(key_value, [])
        if limit is not None:
            # Already resident in RAM either way; slicing just
            # avoids handing an oversized list to the caller.
            return result[:limit]
        return result

    def get_attrs(self, suffix):
        return self.attrs.get(suffix)

    def num_keys(self):
        return sum(len(v) for v in self.postings.values())


# ============================================================
# SQLITE (DISK-BACKED) BACKEND -- with capped fetch + checkpoint
# ============================================================

class SqliteIndex:

    def __init__(self, source_prefix, db_path, manifest_path=None):

        self.source_prefix = source_prefix
        self.db_path = db_path
        self.manifest_path = manifest_path or (str(db_path) + ".manifest.json")
        self.count = 0

        self.conn = sqlite3.connect(db_path)
        self.conn.execute("PRAGMA synchronous = OFF")
        self.conn.execute("PRAGMA journal_mode = WAL")  # survives a crash mid-write

        self.conn.execute(
            "CREATE TABLE IF NOT EXISTS postings "
            "(key_type TEXT, key_value TEXT, suffix TEXT)"
        )
        self.conn.execute(
            "CREATE TABLE IF NOT EXISTS attrs "
            "(suffix TEXT PRIMARY KEY, name TEXT, address TEXT, country TEXT)"
        )
        self.conn.commit()

        self._buffer = []
        self._attr_buffer = []
        self._index_built = False

    # -------- manifest / resume --------

    def _read_manifest(self):
        if os.path.exists(self.manifest_path):
            with open(self.manifest_path, "r", encoding="utf-8") as f:
                return json.load(f)
        return None

    def _write_manifest(self, rows_indexed, complete):
        manifest = {
            "source_prefix": self.source_prefix,
            "rows_indexed": rows_indexed,
            "complete": complete,
        }
        tmp_path = self.manifest_path + ".tmp"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(manifest, f)
        os.replace(tmp_path, self.manifest_path)

    def resume_point(self):
        """
        Returns (rows_already_indexed, already_complete). The
        caller (build_index) uses this to decide how many input
        rows to skip, and whether to skip building this source's
        index at all.
        """
        manifest = self._read_manifest()
        if manifest is None:
            return 0, False
        return manifest.get("rows_indexed", 0), manifest.get("complete", False)

    # -------- ingest --------

    def add_record(self, entity_id, name, address, country):

        _, suffix = split_entity_id(entity_id)
        suffix = str(suffix)

        self._attr_buffer.append((suffix, name, address, country))

        for key_type, key_value in make_name_keys(name) + make_unicode_name_keys(name):
            self._buffer.append((key_type, compose_key(country, key_value), suffix))

        for key_type, key_value in make_address_keys(address):
            self._buffer.append((key_type, compose_key(country, key_value), suffix))

        for key_type, key_value in make_a4_keys(address) + make_at2_keys(address):
            self._buffer.append((key_type, compose_key(country, key_value), suffix))

        self.count += 1

        if len(self._buffer) >= 200_000:
            self._flush()

    def _flush(self):

        if self._buffer:
            self.conn.executemany("INSERT INTO postings VALUES (?,?,?)", self._buffer)
            self._buffer = []

        if self._attr_buffer:
            self.conn.executemany(
                "INSERT OR REPLACE INTO attrs VALUES (?,?,?,?)", self._attr_buffer
            )
            self._attr_buffer = []

        # Commit on every flush (not just at finalize()) so a kill
        # between flushes loses at most one buffer's worth of rows,
        # and record how far we got so a restart knows where to
        # resume from.
        self.conn.commit()
        self._write_manifest(rows_indexed=self.count, complete=False)

    def finalize(self):

        self._flush()

        if not self._index_built:
            self.conn.execute(
                "CREATE INDEX IF NOT EXISTS idx_postings "
                "ON postings(key_type, key_value)"
            )
            self.conn.commit()
            self._index_built = True

        self._write_manifest(rows_indexed=self.count, complete=True)

    # -------- query --------

    def bucket(self, key_type, key_value, limit=None):
        if limit is not None:
            cur = self.conn.execute(
                "SELECT suffix FROM postings WHERE key_type=? AND key_value=? LIMIT ?",
                (key_type, key_value, limit),
            )
        else:
            cur = self.conn.execute(
                "SELECT suffix FROM postings WHERE key_type=? AND key_value=?",
                (key_type, key_value),
            )
        return [row[0] for row in cur.fetchall()]

    def get_attrs(self, suffix):
        cur = self.conn.execute(
            "SELECT name, address, country FROM attrs WHERE suffix=?",
            (str(suffix),),
        )
        return cur.fetchone()

    def num_keys(self):
        cur = self.conn.execute(
            "SELECT COUNT(*) FROM (SELECT DISTINCT key_type, key_value FROM postings)"
        )
        return cur.fetchone()[0]


# ============================================================
# BUILD (with resume)
# ============================================================

def build_index(path, source_prefix, limit=None, backend="memory", db_path=None,
                 manifest_path=None):
    """
    Same signature and behavior as before for backend="memory"
    (no persistence is possible for an in-RAM structure across a
    kernel restart -- if the run dies, a memory-backend index must
    be rebuilt from scratch; use backend="sqlite" for anything
    that needs to survive a Kaggle interruption, per the
    checkpoint/resume plan).

    backend="sqlite" now resumes: if db_path + its manifest show a
    previously COMPLETE build for this source, the existing
    database is reopened and returned immediately (no re-read of
    `path` at all). If the manifest shows a partial build, only
    the remaining rows of `path` are read and appended.
    """

    if backend == "sqlite":
        if db_path is None:
            raise ValueError("db_path is required for the sqlite backend")

        index = SqliteIndex(source_prefix, db_path, manifest_path=manifest_path)
        rows_done, complete = index.resume_point()

        if complete:
            print(
                f"[indexer] {source_prefix} index at {db_path} is already "
                f"complete ({rows_done:,} rows) -- reusing, no rebuild."
            )
            index.count = rows_done
            return index

        if rows_done:
            print(
                f"[indexer] Resuming {source_prefix} index at {db_path}: "
                f"{rows_done:,} rows already indexed, continuing from there."
            )
        index.count = rows_done

    else:
        index = MemoryIndex(source_prefix)
        rows_done = 0

    print(f"[indexer] Building {backend} index for {source_prefix} from: {path}")

    for i, row in enumerate(read_tsv(path)):

        if i < rows_done:
            continue  # already indexed in a previous, interrupted run
            # NOTE: this still parses every skipped row (read_tsv is a
            # plain csv.DictReader generator) -- resume avoids redoing
            # key generation + inserts, which dominate cost, but not
            # the comparatively cheap TSV parse. Acceptable trade-off
            # for correctness/simplicity; revisit only if profiling
            # shows the parse itself is significant.

        if limit is not None and i >= limit:
            break

        index.add_record(
            entity_id=row.get("entity_id", ""),
            name=row.get("business_name", ""),
            address=row.get("business_address", ""),
            country=row.get("country", "").strip(),
        )

        if index.count % 500_000 == 0:
            print(f"[indexer]   indexed {index.count:,} {source_prefix} records...")

    if backend == "sqlite":
        index.finalize()

    print(
        f"[indexer] Finished {source_prefix}: {index.count:,} records, "
        f"{index.num_keys():,} distinct blocking keys."
    )

    return index
''', encoding="utf-8")
pathlib.Path(SRC_DIR, "checkpoint.py").write_text(r'''"""
checkpoint.py
============================================================
Generic checkpoint/resume support for long-running, shard-based
stages of the pipeline (index building, candidate generation +
feature calculation, scoring).

Design goals:
  - A Kaggle session interruption at ANY point must lose at most
    one shard's worth of work, never the whole run.
  - Resuming must be a no-op decision ("skip, already done") for
    every shard that already finished, so restarting the same
    notebook cell is always safe to re-run.
  - No external state beyond plain files already in
    /kaggle/working, so it survives a kernel restart on the same
    persisted working directory.

This module knows nothing about business logic (blocking,
features, etc.) -- it only tracks "which shards of which stage
are done", so it is shared by every stage of the pipeline.

Manifest layout (one JSON file per stage):

    {
      "stage": "candidates_test_full",
      "shard_size": 50000,
      "total_rows": 1730000,
      "completed_shards": [0, 1, 2, ...],
      "updated_at": "2026-09-26T12:00:00"
    }

Each completed shard's OUTPUT is written to its own file
(e.g. candidate_pairs_shard_0007.tsv). On resume, a shard is
only re-run if it is missing from completed_shards, so half-
written output files from a killed process are simply
overwritten -- there is never a "partially valid" shard file.
"""

import json
import os
import tempfile
from datetime import datetime, timezone


def _atomic_write_json(path, data):
    """
    Write JSON atomically: write to a temp file in the same
    directory, then os.replace() it over the target. This means
    a kill -9 mid-write can never leave a corrupt manifest --
    readers always see either the old or the new manifest, never
    a half-written one.
    """
    directory = os.path.dirname(path) or "."
    os.makedirs(directory, exist_ok=True)

    fd, tmp_path = tempfile.mkstemp(dir=directory, prefix=".ckpt_", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f)
        os.replace(tmp_path, path)
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def load_manifest(manifest_path, stage, shard_size, total_rows):
    """
    Load an existing manifest, or create a fresh in-memory one if
    none exists yet or it doesn't match the current run's shape
    (stage name / shard_size / total_rows). A shape mismatch means
    the run configuration changed, so we start over rather than
    silently reusing shard boundaries computed under a different
    shard_size.
    """
    if os.path.exists(manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if (
            data.get("stage") == stage
            and data.get("shard_size") == shard_size
            and data.get("total_rows") == total_rows
        ):
            data["completed_shards"] = set(data.get("completed_shards", []))
            return data

        print(
            f"[checkpoint] manifest at {manifest_path} does not match "
            f"current run shape -- starting a fresh manifest for '{stage}'"
        )

    return {
        "stage": stage,
        "shard_size": shard_size,
        "total_rows": total_rows,
        "completed_shards": set(),
        "updated_at": None,
    }


def save_manifest(manifest_path, manifest):
    to_write = dict(manifest)
    to_write["completed_shards"] = sorted(manifest["completed_shards"])
    to_write["updated_at"] = datetime.now(timezone.utc).isoformat()
    _atomic_write_json(manifest_path, to_write)


def mark_shard_complete(manifest_path, manifest, shard_index):
    """
    Call this AFTER a shard's output file has been fully written
    and closed (renamed into place if you write-then-rename), so
    the manifest can never claim a shard is done while its output
    file is still incomplete.
    """
    manifest["completed_shards"].add(shard_index)
    save_manifest(manifest_path, manifest)


def is_shard_complete(manifest, shard_index):
    return shard_index in manifest["completed_shards"]


def iter_shards(total_rows, shard_size):
    """
    Yield (shard_index, start_row, end_row) triples covering
    [0, total_rows) in fixed-size, deterministic chunks. Using a
    fixed shard_size (rather than "N shards") means adding rows
    to a source file never changes the boundaries of shards that
    already completed.
    """
    shard_index = 0
    start = 0
    while start < total_rows:
        end = min(start + shard_size, total_rows)
        yield shard_index, start, end
        start = end
        shard_index += 1


def shard_output_path(out_dir, prefix, shard_index):
    return os.path.join(out_dir, f"{prefix}_shard_{shard_index:05d}.tsv")


def write_shard_atomically(final_path, write_fn):
    """
    write_fn(tmp_path) must fully write the shard's content to
    tmp_path. This function then renames it into place, so any
    reader only ever sees a complete file at final_path, and a
    process killed mid-write leaves only an orphaned .tmp file
    (never a truncated final_path).
    """
    directory = os.path.dirname(final_path) or "."
    os.makedirs(directory, exist_ok=True)

    tmp_path = final_path + ".tmp"
    write_fn(tmp_path)
    os.replace(tmp_path, final_path)


def concat_completed_shards(out_dir, prefix, manifest, header_line, final_path):
    """
    Convenience for the last step of a stage: stitch every
    completed shard's file into one final output, in shard order.
    Safe to call multiple times (idempotent) since it always
    rewrites final_path from the shard files, and only ever reads
    shards recorded as complete in the manifest.
    """
    shard_indices = sorted(manifest["completed_shards"])

    def _write(tmp_path):
        with open(tmp_path, "w", encoding="utf-8", newline="") as out_f:
            out_f.write(header_line.rstrip("\n") + "\n")
            for shard_index in shard_indices:
                shard_path = shard_output_path(out_dir, prefix, shard_index)
                with open(shard_path, "r", encoding="utf-8") as in_f:
                    next(in_f, None)  # skip that shard's own header
                    for line in in_f:
                        out_f.write(line)

    write_shard_atomically(final_path, _write)
    return final_path
''', encoding="utf-8")

print("Wrote:", ['indexer.py', 'checkpoint.py'])

Wrote: ['indexer.py', 'checkpoint.py']


In [16]:
from indexer import build_index, split_entity_id, join_entity_id, compose_key, discover_dataset_root
import checkpoint as ckpt

index_limit = SMOKE_TEST_INDEX_LIMIT if SMOKE_TEST else None

t0 = time.time()
s2_train_index = build_index(
    S2_TRAIN, "S2",
    limit=index_limit,
    backend=INDEX_BACKEND,
    db_path=str(IDX_DIR / "s2_train.sqlite") if INDEX_BACKEND == "sqlite" else None,
)
print(f"[index] S2 train index ready in {time.time()-t0:.1f}s ({s2_train_index.count:,} records)")

[indexer] Building memory index for S2 from: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
[indexer] Finished S2: 100,000 records, 1,135,542 distinct blocking keys.
[index] S2 train index ready in 13.4s (100,000 records)


## CELL 9 — Build S3 index

In [17]:
t0 = time.time()
s3_train_index = build_index(
    S3_TRAIN, "S3",
    limit=index_limit,
    backend=INDEX_BACKEND,
    db_path=str(IDX_DIR / "s3_train.sqlite") if INDEX_BACKEND == "sqlite" else None,
)
print(f"[index] S3 train index ready in {time.time()-t0:.1f}s ({s3_train_index.count:,} records)")

train_indexes = {"S2": s2_train_index, "S3": s3_train_index}

[indexer] Building memory index for S3 from: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv
[indexer] Finished S3: 100,000 records, 1,186,076 distinct blocking keys.
[index] S3 train index ready in 13.5s (100,000 records)


## CELL 10 — Candidate-generation functions

Writes `candidates.py` (capped `bucket()` lookups, one-line change from Phase 1: `limit=cap+1` so an oversized bucket costs `O(cap)` to detect, not `O(bucket size)`), `features.py` (20 original feature columns unchanged + 4 non-Latin fallback columns, RapidFuzz-backed ratio with an automatic difflib fallback), and `labels.py` (memory-lean `load_ground_truth_compact` for the full 2.2M-row ground truth).

In [18]:
import pathlib
pathlib.Path(SRC_DIR).mkdir(parents=True, exist_ok=True)

pathlib.Path(SRC_DIR, "candidates.py").write_text(r'''"""
candidates.py
============================================================
CHANGE from the original: index.bucket(key_type, composed) is now
called as index.bucket(key_type, composed, limit=cap + 1).

Why cap + 1 and not cap: we still need to distinguish "bucket size
== cap exactly" (keep it) from "bucket size > cap" (drop it), so we
ask for one more row than the cap allows. For MemoryIndex this is
free (already in RAM). For SqliteIndex this turns a potentially
huge, wasted `SELECT ... ` + fetchall() on a hot key into a bounded
`SELECT ... LIMIT cap+1` -- seeSee indexer.py's module docstring
for the full reasoning and why this matters specifically at full
(5M+ row) production scale, not at the 100K-row dev scale the
original numbers were measured on.

Everything else in this file (the candidate dict shape, key_types/
min_freq bookkeeping, generate_candidate_pairs_for_row) is
UNCHANGED.
"""

from blocking import (
    make_name_keys,
    make_unicode_name_keys,
    make_address_keys,
    make_a4_keys,
    make_at2_keys,
    DEFAULT_FREQ_CAPS,
)
from indexer import compose_key


def find_candidates(s1_row, index, freq_caps=None):

    freq_caps = freq_caps or DEFAULT_FREQ_CAPS

    country = s1_row.get("country", "").strip()
    name = s1_row.get("business_name", "")
    address = s1_row.get("business_address", "")

    all_keys = (
        make_name_keys(name)
        + make_unicode_name_keys(name)
        + make_address_keys(address)
        + make_a4_keys(address)
        + make_at2_keys(address)
    )

    candidates = {}

    for key_type, key_value in all_keys:

        cap = freq_caps.get(key_type, DEFAULT_FREQ_CAPS.get(key_type, 500))

        composed = compose_key(country, key_value)

        # Ask for at most cap + 1 rows. If we get back cap + 1, the
        # true bucket is > cap somewhere at or beyond that row, so
        # it's still correctly treated as "too broad" below -- we
        # just never had to materialize the rest of it.
        bucket = index.bucket(key_type, composed, limit=cap + 1)
        bucket_size = len(bucket)

        if bucket_size == 0:
            continue

        if bucket_size > cap:
            continue

        for suffix in bucket:

            entry = candidates.get(suffix)

            if entry is None:
                candidates[suffix] = {
                    "key_types": {key_type},
                    "min_freq": bucket_size,
                }
            else:
                entry["key_types"].add(key_type)
                entry["min_freq"] = min(entry["min_freq"], bucket_size)

    return candidates


def generate_candidate_pairs_for_row(s1_row, indexes_by_source, freq_caps=None):

    s1_id = s1_row.get("entity_id", "")
    rows = []

    for source, index in indexes_by_source.items():

        candidates = find_candidates(s1_row, index, freq_caps=freq_caps)

        for suffix, info in candidates.items():

            candidate_id = f"{source}-{suffix}"

            rows.append(
                {
                    "source1_entity_id": s1_id,
                    "candidate_entity_id": candidate_id,
                    "source": source,
                    "blocking_key_types": "|".join(sorted(info["key_types"])),
                    "blocking_num_keys": len(info["key_types"]),
                    "blocking_min_freq": info["min_freq"],
                }
            )

    return rows
''', encoding="utf-8")
pathlib.Path(SRC_DIR, "features.py").write_text(r'''"""
features.py
============================================================
CHANGES from the original (see PHASE1_ARCHITECTURE_AUDIT.md,
sections "Existing feature set" and "Runtime optimization plan"
for the measurements behind these changes):

1. NON-LATIN FIX (correctness, not just speed):
   name_char_similarity / name_levenshtein_similarity / etc. used
   to be computed ONLY on normalize_business_name()/normalize_address(),
   which strip non-Latin scripts to "". For a true-match pair where
   both sides are in Devanagari/Telugu/Kannada/Bengali, every
   similarity feature silently became 0 or the "missing" default,
   even though the strings are genuinely very similar. This module
   now also computes the same feature suite on a script-agnostic
   normalization (normalize.normalize_unicode_preserving) and uses
   whichever normalization actually has content, so the ML model
   gets real signal on non-Latin rows instead of a constant.

2. SPEED: the original had TWO independent implementations of
   "edit-distance-based ratio" per field (text_similarity() via
   difflib.SequenceMatcher, and levenshtein_similarity() via
   rapidfuzz). difflib's SequenceMatcher is pure Python and is the
   single most expensive call in the old feature set -- at the
   ~440M-pair scale estimated for the full test set, removing one
   of the two redundant passes per field is a meaningful win.
   rapidfuzz.fuzz.ratio() is a compatible-by-design, C-accelerated
   reimplementation of the same "difflib-style" ratio, so
   name_char_similarity / address_char_similarity are now computed
   with it too. If rapidfuzz is unavailable, this falls back to the
   original difflib implementation exactly as before -- no feature
   changes shape, only which backend computes it.

3. Feature list is APPENDED to, not reordered or removed, so
   existing train_features_*.tsv files stay a strict prefix-
   compatible subset of the new schema (old columns keep the same
   name and meaning).
"""

from functools import lru_cache

from normalize import (
    normalize_business_name,
    normalize_address,
    normalize_business_name_script_agnostic,
    normalize_unicode_preserving,
    is_non_latin,
)

from similarity import token_similarity, address_number_match, country_match


# --------------------------------------------------------------
# Ratio backend: rapidfuzz preferred (C-accelerated, difflib-
# compatible ratio), pure-Python difflib as a guaranteed fallback.
# One implementation now serves BOTH "char_similarity" and
# "levenshtein_similarity" -- see change #2 above.
# --------------------------------------------------------------

try:
    from rapidfuzz.fuzz import ratio as _rapidfuzz_ratio

    def _ratio(a, b):
        if not a or not b:
            return 0.0
        return _rapidfuzz_ratio(a, b) / 100.0

except ImportError:

    from difflib import SequenceMatcher

    def _ratio(a, b):
        if not a or not b:
            return 0.0
        return SequenceMatcher(None, a, b).ratio()


levenshtein_similarity = _ratio  # kept as a public name for compatibility


@lru_cache(maxsize=2_000_000)
def cached_normalize_business_name(value):
    return normalize_business_name(value)


@lru_cache(maxsize=2_000_000)
def cached_normalize_address(value):
    return normalize_address(value)


@lru_cache(maxsize=2_000_000)
def cached_normalize_business_name_unicode(value):
    return normalize_business_name_script_agnostic(value)


@lru_cache(maxsize=2_000_000)
def cached_normalize_address_unicode(value):
    return normalize_unicode_preserving(value)


# --------------------------------------------------------------
# Character n-gram Jaccard
# --------------------------------------------------------------

def char_ngram_jaccard(a, b, n=2):

    if not a or not b:
        return 0.0

    if len(a) < n or len(b) < n:
        return 0.0

    grams_a = {a[i:i + n] for i in range(len(a) - n + 1)}
    grams_b = {b[i:i + n] for i in range(len(b) - n + 1)}

    if not grams_a or not grams_b:
        return 0.0

    return len(grams_a & grams_b) / len(grams_a | grams_b)


def prefix_similarity(a, b, k=4):

    if not a or not b:
        return 0.0

    a_p, b_p = a[:k], b[:k]
    max_len = min(len(a_p), len(b_p))

    if max_len == 0:
        return 0.0

    matched = sum(1 for x, y in zip(a_p, b_p) if x == y)
    return matched / max_len


def suffix_similarity(a, b, k=4):

    if not a or not b:
        return 0.0

    a_s, b_s = a[-k:], b[-k:]
    max_len = min(len(a_s), len(b_s))

    if max_len == 0:
        return 0.0

    matched = sum(1 for x, y in zip(reversed(a_s), reversed(b_s)) if x == y)
    return matched / max_len


def token_overlap_count(a, b):

    if not a or not b:
        return 0

    return len(set(a.split()) & set(b.split()))


# --------------------------------------------------------------
# Feature columns
# --------------------------------------------------------------
# Original columns are UNCHANGED in name, meaning and position.
# New columns are appended at the end.

FEATURE_COLUMNS = [

    # Name features (original)
    "name_exact_match",
    "name_char_similarity",
    "name_levenshtein_similarity",
    "name_token_jaccard",
    "name_token_overlap",
    "name_compact_similarity",
    "name_prefix_similarity",
    "name_suffix_similarity",
    "name_char_ngram_jaccard",

    # Address features (original)
    "address_exact_match",
    "address_char_similarity",
    "address_levenshtein_similarity",
    "address_token_jaccard",
    "address_token_overlap",
    "address_number_match",
    "address_char_ngram_jaccard",

    # Other (original)
    "country_match",
    "name_missing",
    "address_missing",
    "combined_name_address_score",

    # ---- NEW: non-Latin fallback signal ----
    "name_is_non_latin",
    "address_is_non_latin",
    "name_char_similarity_unicode",
    "name_token_jaccard_unicode",
]


def calculate_features(name1, address1, country1, name2, address2, country2):

    # -------- ASCII-folded normalization (original path) --------
    n1 = cached_normalize_business_name(name1)
    n2 = cached_normalize_business_name(name2)
    a1 = cached_normalize_address(address1)
    a2 = cached_normalize_address(address2)

    name_missing = int(not n1 or not n2)
    address_missing = int(not a1 or not a2)

    name_char_sim = _ratio(n1, n2)
    name_lev_sim = _ratio(n1, n2)  # same backend now, see module docstring #2
    name_tok_jaccard = token_similarity(n1, n2)
    name_compact_sim = _ratio(n1.replace(" ", ""), n2.replace(" ", ""))

    address_char_sim = _ratio(a1, a2)
    address_lev_sim = _ratio(a1, a2)
    address_tok_jaccard = token_similarity(a1, a2)

    combined = 0.6 * name_char_sim + 0.4 * address_char_sim

    # -------- NEW: script-agnostic fallback for non-Latin names --------
    # Only pay for the extra normalization/similarity work when the
    # ASCII-folded form actually lost content -- cheap for the
    # (majority) Latin-script rows, since is_non_latin() short-
    # circuits on the already-cached ascii_form.
    name_non_latin = is_non_latin(name1) or is_non_latin(name2)

    if name_non_latin:
        un1 = cached_normalize_business_name_unicode(name1)
        un2 = cached_normalize_business_name_unicode(name2)
        name_char_sim_unicode = _ratio(un1, un2)
        name_tok_jaccard_unicode = token_similarity(un1, un2)
    else:
        name_char_sim_unicode = name_char_sim
        name_tok_jaccard_unicode = name_tok_jaccard

    address_non_latin = is_non_latin(address1) or is_non_latin(address2)

    return {

        "name_exact_match": int(bool(n1) and n1 == n2),
        "name_char_similarity": name_char_sim,
        "name_levenshtein_similarity": name_lev_sim,
        "name_token_jaccard": name_tok_jaccard,
        "name_token_overlap": token_overlap_count(n1, n2),
        "name_compact_similarity": name_compact_sim,
        "name_prefix_similarity": prefix_similarity(n1, n2),
        "name_suffix_similarity": suffix_similarity(n1, n2),
        "name_char_ngram_jaccard": char_ngram_jaccard(n1, n2),

        "address_exact_match": int(bool(a1) and a1 == a2),
        "address_char_similarity": address_char_sim,
        "address_levenshtein_similarity": address_lev_sim,
        "address_token_jaccard": address_tok_jaccard,
        "address_token_overlap": token_overlap_count(a1, a2),
        "address_number_match": address_number_match(address1, address2),
        "address_char_ngram_jaccard": char_ngram_jaccard(a1, a2),

        "country_match": country_match(country1, country2),
        "name_missing": name_missing,
        "address_missing": address_missing,
        "combined_name_address_score": combined,

        "name_is_non_latin": int(name_non_latin),
        "address_is_non_latin": int(address_non_latin),
        "name_char_similarity_unicode": name_char_sim_unicode,
        "name_token_jaccard_unicode": name_tok_jaccard_unicode,
    }
''', encoding="utf-8")
pathlib.Path(SRC_DIR, "labels.py").write_text(r'''"""
labels.py
============================================================
PRESERVED: load_ground_truth() / label_for() are UNCHANGED and
still correct -- keep using them for train_10k-scale work where
their extra convenience (pre-split "S2"/"S3"/"all" sets) is cheap.

ADDED: load_ground_truth_compact(), a memory-lean alternative for
the full 2.2M-row train_ground_truth.tsv.

MEASURED (this sandbox, 3.9 GB total RAM, Python 3.12):
  load_ground_truth() on the full file -> 2,206,821 entries,
  peak RSS ~2.84 GB, ~25s.

That is the ENTIRE per-S1-row ground truth structure sitting in
RAM before candidate generation, feature calculation, indexes, or
anything else has been loaded -- on a Kaggle instance with, say,
13-16 GB total RAM, this is a meaningful fixed cost, and on a
smaller instance it can be the difference between fitting in
memory and an OOM kill mid-run (which is exactly what truncated a
larger validation run attempted during this audit).

The cost comes from storing THREE Python sets per S1 row
("S2", "S3", "all") instead of one -- each empty/small set has
real fixed overhead (hash table allocation) independent of how
many ids it holds, so building three per row roughly triples the
per-row container overhead versus storing the raw id set once and
filtering by prefix on demand (which is O(len(entry)), and entries
have on average only a handful of matches).

load_ground_truth_compact() stores exactly one frozenset per row.
matches_for_source()/label_for_compact() below reconstruct the
"S2 only" / "S3 only" view on demand, at effectively no extra cost
since per-row match counts are small (median well under 10, per
analyze_ground_truth.py's own distribution check).
"""

import csv


def load_ground_truth(path):
    """
    UNCHANGED. Returns dict: source1_entity_id -> {
        "S2": set of matched S2 ids,
        "S3": set of matched S3 ids,
        "all": set of every matched id (any source),
    }
    """

    ground_truth = {}

    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:

        reader = csv.DictReader(f, delimiter="\t")

        for row in reader:

            s1_id = row["source1_entity_id"]
            raw = (row.get("matched_entity_ids") or "").strip()

            if raw:
                all_ids = {x.strip() for x in raw.split(",") if x.strip()}
            else:
                all_ids = set()

            ground_truth[s1_id] = {
                "S2": {x for x in all_ids if x.startswith("S2-")},
                "S3": {x for x in all_ids if x.startswith("S3-")},
                "all": all_ids,
            }

    return ground_truth


def label_for(s1_id, candidate_id, ground_truth):
    entry = ground_truth.get(s1_id)
    if not entry:
        return 0
    return int(candidate_id in entry["all"])


# ============================================================
# NEW: memory-lean loader for full-scale (2.2M+ row) ground truth
# ============================================================

_EMPTY_FROZENSET = frozenset()


def load_ground_truth_compact(path):
    """
    Returns dict: source1_entity_id -> frozenset of matched ids
    (mixed S2-*/S3-*). One container per row instead of three.

    frozenset (vs set) is used because these are never mutated
    after loading, and CPython's frozenset has slightly less
    per-object overhead than a mutable set for read-only use.
    """

    ground_truth = {}

    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:

        reader = csv.DictReader(f, delimiter="\t")

        for row in reader:

            s1_id = row["source1_entity_id"]
            raw = (row.get("matched_entity_ids") or "").strip()

            if raw:
                ground_truth[s1_id] = frozenset(
                    x.strip() for x in raw.split(",") if x.strip()
                )
            else:
                ground_truth[s1_id] = _EMPTY_FROZENSET

    return ground_truth


def matches_for_source(entry, source_prefix):
    """
    entry: a frozenset from load_ground_truth_compact(). Filters
    by prefix on demand -- cheap, since entry is typically only a
    handful of ids (see analyze_ground_truth.py's match_count
    distribution: median match count is small).
    """
    if not entry:
        return _EMPTY_FROZENSET
    return frozenset(x for x in entry if x.startswith(source_prefix))


def label_for_compact(s1_id, candidate_id, ground_truth):
    entry = ground_truth.get(s1_id)
    if not entry:
        return 0
    return int(candidate_id in entry)
''', encoding="utf-8")

print("Wrote:", ['candidates.py', 'features.py', 'labels.py'])

Wrote: ['candidates.py', 'features.py', 'labels.py']


In [19]:
from candidates import find_candidates, generate_candidate_pairs_for_row
from features import calculate_features, FEATURE_COLUMNS
from labels import load_ground_truth_compact, matches_for_source, label_for_compact

CANDIDATE_COLUMNS = [
    "source1_entity_id", "candidate_entity_id", "source",
    "blocking_key_types", "blocking_num_keys", "blocking_min_freq",
]
FEATURE_FILE_COLUMNS = CANDIDATE_COLUMNS + FEATURE_COLUMNS

print("[candidates] CANDIDATE_COLUMNS:", CANDIDATE_COLUMNS)
print("[features]   FEATURE_COLUMNS (", len(FEATURE_COLUMNS), "):", FEATURE_COLUMNS)

_example = next(read_tsv(S1_TRAIN))
_cands = generate_candidate_pairs_for_row(_example, train_indexes)
print(f"\n[candidates] example S1 row produced {len(_cands)} candidate pairs; first 3:")
for c in _cands[:3]:
    print("   ", c)

[candidates] CANDIDATE_COLUMNS: ['source1_entity_id', 'candidate_entity_id', 'source', 'blocking_key_types', 'blocking_num_keys', 'blocking_min_freq']
[features]   FEATURE_COLUMNS ( 24 ): ['name_exact_match', 'name_char_similarity', 'name_levenshtein_similarity', 'name_token_jaccard', 'name_token_overlap', 'name_compact_similarity', 'name_prefix_similarity', 'name_suffix_similarity', 'name_char_ngram_jaccard', 'address_exact_match', 'address_char_similarity', 'address_levenshtein_similarity', 'address_token_jaccard', 'address_token_overlap', 'address_number_match', 'address_char_ngram_jaccard', 'country_match', 'name_missing', 'address_missing', 'combined_name_address_score', 'name_is_non_latin', 'address_is_non_latin', 'name_char_similarity_unicode', 'name_token_jaccard_unicode']

[candidates] example S1 row produced 136 candidate pairs; first 3:
    {'source1_entity_id': 'S1-925783039', 'candidate_entity_id': 'S2-361957120', 'source': 'S2', 'blocking_key_types': 'NP6|UNP4|UNP6', 'blo

## CELL 11 — Training candidate generation

Single sequential pass over the **full** `train_source1.tsv`, sharded and checkpointed by S1 row range (`CANDIDATE_SHARD_SIZE` rows/shard). For every shard: generate candidates against the S2 and S3 indexes, immediately fold in recall accounting (needs no second pass over the data) and sample missed matches, then atomically write the shard's candidate-pairs file + a small per-shard stats JSON, and only *then* mark the shard complete. A shard is only ever (re)computed once — on resume, already-complete shards are skipped (their input rows are still parsed, which is cheap, but no candidate generation/feature work is redone).

In [20]:
def build_gt_lookup(gt_path):
    if gt_path is None:
        return None
    return load_ground_truth_compact(gt_path)

def run_candidate_generation_stage(
    s1_path, indexes, total_s1_rows, shard_size, out_dir, stage_name,
    gt=None, s1_limit=None, missed_sample_cap_per_shard=200,
):
    """
    One sequential pass over s1_path. Streams candidate generation,
    writes one candidate-pairs shard file + one stats JSON file per
    shard, checkpointed via checkpoint.py so a kill mid-run only costs
    the in-flight shard.

    Returns the manifest dict.
    """
    effective_total = total_s1_rows if s1_limit is None else min(total_s1_rows, s1_limit)
    manifest_path = str(CKPT_DIR / f"manifest_{stage_name}.json")
    manifest = ckpt.load_manifest(manifest_path, stage_name, shard_size, effective_total)

    has_labels = gt is not None
    sources = list(indexes.keys())

    cand_buffer = []
    missed_buffer = []
    shard_stats = {s: {"actual": 0, "found": 0} for s in sources}
    shard_combined = {"actual": 0, "found": 0}
    cand_counts = []

    def flush(shard_idx):
        cand_path = ckpt.shard_output_path(out_dir, f"cand_{stage_name}", shard_idx)
        def _write_cand(tmp):
            with open(tmp, "w", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=CANDIDATE_COLUMNS, delimiter="\t")
                w.writeheader()
                for r in cand_buffer:
                    w.writerow(r)
        ckpt.write_shard_atomically(cand_path, _write_cand)

        if has_labels:
            missed_path = ckpt.shard_output_path(out_dir, f"missed_{stage_name}", shard_idx)
            def _write_missed(tmp):
                with open(tmp, "w", newline="", encoding="utf-8") as f:
                    w = csv.writer(f, delimiter="\t")
                    w.writerow(["source1_entity_id", "missed_entity_id", "source"])
                    for row in missed_buffer:
                        w.writerow(row)
            ckpt.write_shard_atomically(missed_path, _write_missed)

        stats_path = os.path.join(out_dir, f"stats_{stage_name}_shard_{shard_idx:05d}.json")
        stats_obj = {"stats": shard_stats, "combined": shard_combined, "cand_counts": cand_counts}
        with open(stats_path + ".tmp", "w", encoding="utf-8") as f:
            json.dump(stats_obj, f)
        os.replace(stats_path + ".tmp", stats_path)

        ckpt.mark_shard_complete(manifest_path, manifest, shard_idx)

    t0 = time.time()
    processed_new = 0
    for i, s1_row in enumerate(read_tsv(s1_path)):
        if i >= effective_total:
            break

        shard_idx = i // shard_size

        if ckpt.is_shard_complete(manifest, shard_idx):
            continue  # already fully processed in a previous run

        s1_id = s1_row.get("entity_id", "")
        pairs = generate_candidate_pairs_for_row(s1_row, indexes, freq_caps=DEFAULT_FREQ_CAPS)
        cand_buffer.extend(pairs)
        cand_counts.append(len(pairs))
        processed_new += 1

        if has_labels:
            by_source = {s: set() for s in sources}
            for p in pairs:
                by_source[p["source"]].add(p["candidate_entity_id"])

            entry = gt.get(s1_id, frozenset())
            for source in sources:
                true_matches = matches_for_source(entry, source)
                found = true_matches & by_source[source]
                shard_stats[source]["actual"] += len(true_matches)
                shard_stats[source]["found"] += len(found)
                for m in (true_matches - found):
                    if len(missed_buffer) < missed_sample_cap_per_shard:
                        missed_buffer.append((s1_id, m, source))

            found_all = entry & set().union(*by_source.values())
            shard_combined["actual"] += len(entry)
            shard_combined["found"] += len(found_all)

        is_last_row_of_shard = ((i + 1) % shard_size == 0)
        is_last_row_overall = (i + 1 == effective_total)
        if is_last_row_of_shard or is_last_row_overall:
            flush(shard_idx)
            cand_buffer, missed_buffer = [], []
            shard_stats = {s: {"actual": 0, "found": 0} for s in sources}
            shard_combined = {"actual": 0, "found": 0}
            cand_counts = []

        if (i + 1) % 100_000 == 0:
            elapsed = time.time() - t0
            print(f"[{stage_name}] scanned {i+1:,}/{effective_total:,} S1 rows "
                  f"({processed_new:,} newly processed) | elapsed {elapsed:.1f}s")

    n_shards_total = len(list(ckpt.iter_shards(effective_total, shard_size)))
    n_shards_done = len(manifest["completed_shards"])
    print(f"[{stage_name}] done: {n_shards_done}/{n_shards_total} shards complete "
          f"({processed_new:,} rows newly processed this run, {time.time()-t0:.1f}s)")
    return manifest


gt_train = build_gt_lookup(GT_TRAIN)
print(f"[labels] loaded ground truth for {len(gt_train):,} S1 entities")

train_s1_limit = SMOKE_TEST_S1_LIMIT if SMOKE_TEST else None

train_cand_manifest = run_candidate_generation_stage(
    S1_TRAIN, train_indexes, N_S1_TRAIN, CANDIDATE_SHARD_SIZE,
    str(SHARD_DIR), "train_candidates",
    gt=gt_train, s1_limit=train_s1_limit,
)

[labels] loaded ground truth for 2,206,821 S1 entities
[train_candidates] done: 1/1 shards complete (2,000 rows newly processed this run, 3.3s)


## CELL 12 — Candidate recall evaluation

Aggregates the per-shard stats JSON files written by CELL 11 -- no re-scan of the data. Reports S2/S3/combined actual/found/missed/recall, plus the candidates/S1 distribution (mean/median/p95/p99/max) and the largest candidate buckets, as required by the brief before any ML scoring happens.

In [21]:
def aggregate_shard_stats(out_dir, stage_name, manifest, sources):
    total_stats = {s: {"actual": 0, "found": 0} for s in sources}
    total_combined = {"actual": 0, "found": 0}
    all_cand_counts = []
    total_pairs = 0

    for shard_idx in sorted(manifest["completed_shards"]):
        stats_path = os.path.join(out_dir, f"stats_{stage_name}_shard_{shard_idx:05d}.json")
        with open(stats_path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        for s in sources:
            total_stats[s]["actual"] += obj["stats"][s]["actual"]
            total_stats[s]["found"] += obj["stats"][s]["found"]
        total_combined["actual"] += obj["combined"]["actual"]
        total_combined["found"] += obj["combined"]["found"]
        all_cand_counts.extend(obj["cand_counts"])
        total_pairs += sum(obj["cand_counts"])

    return total_stats, total_combined, all_cand_counts, total_pairs


def percentile(sorted_list, p):
    if not sorted_list:
        return 0
    idx = min(len(sorted_list) - 1, int(round(p * (len(sorted_list) - 1))))
    return sorted_list[idx]


train_stats, train_combined, train_cand_counts, train_total_pairs = aggregate_shard_stats(
    str(SHARD_DIR), "train_candidates", train_cand_manifest, list(train_indexes.keys())
)

print("=" * 70)
print("CANDIDATE RECALL (TRAIN)")
print("=" * 70)
for source in ("S2", "S3"):
    a, fo = train_stats[source]["actual"], train_stats[source]["found"]
    missed = a - fo
    recall = (fo / a) if a else float("nan")
    print(f"{source} actual matches : {a:,}")
    print(f"{source} candidates found: {fo:,}")
    print(f"{source} missed         : {missed:,}")
    print(f"{source} recall         : {recall:.4%}")
    print()

c_a, c_f = train_combined["actual"], train_combined["found"]
c_missed = c_a - c_f
c_recall = (c_f / c_a) if c_a else float("nan")
print(f"Combined actual matches : {c_a:,}")
print(f"Combined found          : {c_f:,}")
print(f"Combined missed         : {c_missed:,}")
print(f"Combined recall         : {c_recall:.4%}")

print("\n" + "=" * 70)
print("CANDIDATES / S1 DISTRIBUTION (TRAIN)")
print("=" * 70)
n_s1 = len(train_cand_counts)
sorted_counts = sorted(train_cand_counts)
avg = (train_total_pairs / n_s1) if n_s1 else 0.0
print(f"number of S1 entities   : {n_s1:,}")
print(f"candidate pairs         : {train_total_pairs:,}")
print(f"average candidates/S1   : {avg:.2f}")
print(f"median candidates/S1    : {percentile(sorted_counts, 0.50):,}")
print(f"95th pct candidates/S1  : {percentile(sorted_counts, 0.95):,}")
print(f"99th pct candidates/S1  : {percentile(sorted_counts, 0.99):,}")
print(f"maximum candidates/S1   : {(sorted_counts[-1] if sorted_counts else 0):,}")

RECALL_REPORT = {
    "train": {
        "by_source": train_stats,
        "combined": train_combined,
        "n_s1": n_s1,
        "total_pairs": train_total_pairs,
        "avg_per_s1": avg,
        "median_per_s1": percentile(sorted_counts, 0.50),
        "p95_per_s1": percentile(sorted_counts, 0.95),
        "p99_per_s1": percentile(sorted_counts, 0.99),
        "max_per_s1": (sorted_counts[-1] if sorted_counts else 0),
    }
}
with open(OUT_DIR / "candidate_recall_report_train.json", "w") as f:
    json.dump(RECALL_REPORT, f, indent=2)
print("\n[saved]", OUT_DIR / "candidate_recall_report_train.json")

if c_recall < 0.95:
    print("\n*** WARNING: combined recall below 95% -- consider retuning DEFAULT_FREQ_CAPS "
          "or adding a blocking rule (see CELL 13) before locking the train/val split. ***")

CANDIDATE RECALL (TRAIN)
S2 actual matches : 3,381
S2 candidates found: 63
S2 missed         : 3,318
S2 recall         : 1.8634%

S3 actual matches : 3,605
S3 candidates found: 63
S3 missed         : 3,542
S3 recall         : 1.7476%

Combined actual matches : 6,986
Combined found          : 126
Combined missed         : 6,860
Combined recall         : 1.8036%

CANDIDATES / S1 DISTRIBUTION (TRAIN)
number of S1 entities   : 2,000
candidate pairs         : 470,938
average candidates/S1   : 235.47
median candidates/S1    : 192
95th pct candidates/S1  : 596
99th pct candidates/S1  : 847
maximum candidates/S1   : 1,063

[saved] /kaggle/working/amazon_ml_solution/output/candidate_recall_report_train.json

*** WARNING: combined recall below 95% -- consider retuning DEFAULT_FREQ_CAPS or adding a blocking rule (see CELL 13) before locking the train/val split. ***


## CELL 13 — Missed-match analysis

Concatenates the per-shard missed-match samples (bounded per shard so this never grows unbounded) and inspects a handful of real misses to sanity-check where recall is being lost -- e.g. genuinely corrupted records vs. a blocking gap worth addressing.

In [22]:
def concat_missed_samples(out_dir, stage_name, manifest, max_total=2000):
    rows = []
    for shard_idx in sorted(manifest["completed_shards"]):
        missed_path = ckpt.shard_output_path(out_dir, f"missed_{stage_name}", shard_idx)
        if not os.path.exists(missed_path):
            continue
        with open(missed_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                rows.append(row)
                if len(rows) >= max_total:
                    return rows
    return rows

missed_samples = concat_missed_samples(str(SHARD_DIR), "train_candidates", train_cand_manifest)
print(f"[missed] collected {len(missed_samples):,} sample missed matches")

missed_path = OUT_DIR / "missed_matches_sample.tsv"
with open(missed_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["source1_entity_id", "missed_entity_id", "source"], delimiter="\t")
    w.writeheader()
    for row in missed_samples:
        w.writerow(row)
print("[saved]", missed_path)

# Inspect a few misses directly against the raw records.
s1_lookup_for_inspection = {}
for row in read_tsv(S1_TRAIN):
    if row["entity_id"] in {m["source1_entity_id"] for m in missed_samples[:20]}:
        s1_lookup_for_inspection[row["entity_id"]] = row
    if len(s1_lookup_for_inspection) >= 20:
        break

print("\nSample missed pairs (first 10):")
for m in missed_samples[:10]:
    s1_id = m["source1_entity_id"]
    miss_id = m["missed_entity_id"]
    source = m["source"]
    _, suffix = split_entity_id(miss_id)
    attrs = train_indexes[source].get_attrs(suffix)
    s1_row = s1_lookup_for_inspection.get(s1_id)
    print("-" * 60)
    print("S1 :", s1_id, "|", s1_row.get("business_name") if s1_row else "?", "|",
          s1_row.get("business_address") if s1_row else "?")
    print(f"{source} :", miss_id, "|", attrs[0] if attrs else "?", "|", attrs[1] if attrs else "?")

[missed] collected 200 sample missed matches
[saved] /kaggle/working/amazon_ml_solution/output/missed_matches_sample.tsv

Sample missed pairs (first 10):
------------------------------------------------------------
S1 : S1-925783039 | Orelee's Barbershop | 1795 Westchester Drive, High Point, NC
S2 : S2-517291332 | ? | ?
------------------------------------------------------------
S1 : S1-925783039 | Orelee's Barbershop | 1795 Westchester Drive, High Point, NC
S2 : S2-157377754 | ? | ?
------------------------------------------------------------
S1 : S1-925783039 | Orelee's Barbershop | 1795 Westchester Drive, High Point, NC
S3 : S3-997698194 | ? | ?
------------------------------------------------------------
S1 : S1-925783039 | Orelee's Barbershop | 1795 Westchester Drive, High Point, NC
S3 : S3-698172821 | ? | ?
------------------------------------------------------------
S1 : S1-773889195 | Prime Money | 17560 Ellis Road, Tahlequah, OK
S2 : S2-970528089 | ? | ?
---------------------

## CELL 14 — Feature precomputation

Two large avoidable costs are removed before feature generation runs at scale:

1. **Repeated S1 lookups.** `candidate_pairs` shards were written in S1 file order (CELL 11 streams `train_source1.tsv`/`test_source1.tsv` sequentially), so the candidate-pair rows for a given S1 id are always contiguous and appear in the same order as the S1 file itself. `S1AttrsStream` exploits this: it walks `read_tsv(s1_path)` **once**, in lockstep with the candidate-pair rows being consumed, instead of loading the full multi-million-row S1 file into a dict or re-reading it per pair. This keeps memory at O(1) regardless of S1 file size.

2. **Repeated normalization of identical strings.** `features.py`'s `cached_normalize_*` functions are already `lru_cache`-wrapped (Phase 1) -- this cell just documents/validates that and is where you would tune `lru_cache(maxsize=...)` in `features.py` if profiling shows cache thrashing at full S2/S3 scale (10M+ combined distinct strings versus a 2M-entry cache).

Also defines the **deterministic** train/validation split function used by CELL 15 (feature sampling) and formalized in CELL 16.

In [23]:
import hashlib

class S1AttrsStream:
    """
    Streams (name, address, country) for S1 ids assuming callers query
    ids in the same (non-decreasing) order they appear in s1_path --
    true here because candidate-pair shards are written in S1 file
    order. Falls back to a small side dict for any (rare/never
    expected) out-of-order id, so correctness never depends on the
    ordering assumption, only speed does.
    """

    def __init__(self, s1_path):
        self._gen = read_tsv(s1_path)
        self._current_id = None
        self._current_attrs = None
        self._exhausted = False
        self._late_lookup_cache = {}

    def _advance_to(self, target_id):
        while not self._exhausted:
            try:
                row = next(self._gen)
            except StopIteration:
                self._exhausted = True
                break
            rid = row.get("entity_id", "")
            attrs = (row.get("business_name", ""), row.get("business_address", ""), row.get("country", ""))
            if rid == target_id:
                self._current_id, self._current_attrs = rid, attrs
                return attrs
            self._current_id, self._current_attrs = rid, attrs
        return None

    def get(self, s1_id):
        if s1_id == self._current_id:
            return self._current_attrs
        attrs = self._advance_to(s1_id)
        if attrs is not None:
            return attrs
        # Out-of-order id (shouldn't happen given file-order sharding) -- safe fallback.
        return self._late_lookup_cache.get(s1_id)


def s1_split_bucket(s1_id, n_buckets=100):
    """Deterministic, stable hash -> [0, n_buckets). Same id always lands
    in the same bucket across reruns/sessions (unlike random.shuffle)."""
    h = hashlib.md5(s1_id.encode("utf-8")).hexdigest()
    return int(h[:8], 16) % n_buckets

VALIDATION_FRACTION = 0.20
VALIDATION_BUCKET_CUTOFF = int(VALIDATION_FRACTION * 100)  # buckets [0, cutoff) -> validation

def is_validation_s1(s1_id):
    return s1_split_bucket(s1_id) < VALIDATION_BUCKET_CUTOFF


# Warm the normalization caches once for every distinct S1 name/address --
# de-duplicates work for S1 entities that recur in many candidate pairs
# (every S1 shows up in ~avg-candidates-per-S1 rows) before the heavier
# per-pair feature loop in CELL 15.
from features import cached_normalize_business_name, cached_normalize_address

t0 = time.time()
_warmed = 0
for row in read_tsv(S1_TRAIN):
    cached_normalize_business_name(row.get("business_name", ""))
    cached_normalize_address(row.get("business_address", ""))
    _warmed += 1
    if SMOKE_TEST and _warmed >= SMOKE_TEST_S1_LIMIT:
        break
print(f"[precompute] warmed normalization caches for {_warmed:,} S1 records in {time.time()-t0:.1f}s")
print(f"[precompute] validation fraction: {VALIDATION_FRACTION:.0%} "
      f"(buckets 0-{VALIDATION_BUCKET_CUTOFF-1} of 100)")

[precompute] warmed normalization caches for 2,000 S1 records in 0.1s
[precompute] validation fraction: 20% (buckets 0-19 of 100)


## CELL 15 — Training feature generation

Processes the candidate-pair shards from CELL 11, **one feature shard per candidate shard** (same shard index -- this keeps every S1 entity's full set of candidate rows inside a single shard, which CELL 16-19 need for correct per-S1 macro-F0.5 grouping). For every candidate pair:

1. Look up its label with `label_for_compact` -- an O(1) set-membership check, not a similarity computation.
2. If it's a **validation-S1** pair (per the deterministic split from CELL 14): always compute features and keep it, so threshold calibration in CELL 18 sees the true, undownsampled precision/recall for every validation entity.
3. Otherwise: always keep positives; keep negatives with a fixed probability (`TRAIN_NEGATIVE_KEEP_PROB` / `VALIDATION_NEGATIVE_KEEP_PROB`, sized below from the actual positive/negative counts measured in CELL 12) so the final matrices have a bounded, memory-friendly size regardless of how large the full dataset is. Singleton (no-true-match) S1 entities in the validation split are always fully kept, never downsampled, because CELL 18/19 need their complete candidate set to check the threshold correctly abstains on them. The keep/drop decision is a **deterministic hash** of the pair id (not `random.random()`), so re-running an interrupted shard reproduces exactly the same sample -- required for checkpoint correctness.

`calculate_features()` (RapidFuzz-backed, with the non-Latin fallback) is only ever called for rows that survive this filter, which is most of the wasted cost avoided for the ~98%+ of candidate pairs that are negatives.

In [24]:
MAX_TRAIN_NEGATIVES = 3_000_000       # bound on how many negative TRAIN rows to materialize
MAX_VALIDATION_NEGATIVES = 6_000_000  # bound on negative VALIDATION rows (kept much larger than
                                       # train's cap since these drive threshold calibration --
                                       # raise this if your instance has the RAM/time budget for
                                       # closer-to-full-fidelity calibration)

# Rough split of the (S1-level, not row-level) recall counts by validation
# fraction to size each pool's negative population for the keep-probability
# calculation below. This is an estimate (assumes negatives are distributed
# roughly evenly across S1 entities) purely to size the sampling rate --
# it does not affect which S1 entities are actually in which split.
_val_frac = VALIDATION_FRACTION
positives_total_train_only = train_combined["found"] * (1 - _val_frac)
positives_total_val_only = train_combined["found"] * _val_frac
negatives_total_train_only = (train_total_pairs - train_combined["found"]) * (1 - _val_frac)
negatives_total_val_only = (train_total_pairs - train_combined["found"]) * _val_frac

TRAIN_NEGATIVE_KEEP_PROB = (
    1.0 if negatives_total_train_only <= 0
    else min(1.0, MAX_TRAIN_NEGATIVES / negatives_total_train_only)
)
VALIDATION_NEGATIVE_KEEP_PROB = (
    1.0 if negatives_total_val_only <= 0
    else min(1.0, MAX_VALIDATION_NEGATIVES / negatives_total_val_only)
)
print(f"[sampling] TRAIN_NEGATIVE_KEEP_PROB      : {TRAIN_NEGATIVE_KEEP_PROB:.6f} "
      f"(estimated population {negatives_total_train_only:,.0f})")
print(f"[sampling] VALIDATION_NEGATIVE_KEEP_PROB : {VALIDATION_NEGATIVE_KEEP_PROB:.6f} "
      f"(estimated population {negatives_total_val_only:,.0f})")
print("[sampling] positives are always kept in both splits.")

def deterministic_keep(pair_key, keep_prob, salt="neg_sample_v1"):
    if keep_prob >= 1.0:
        return True
    h = hashlib.md5(f"{salt}:{pair_key}".encode("utf-8")).hexdigest()
    bucket = int(h[:8], 16) / 0xFFFFFFFF
    return bucket < keep_prob


def run_feature_generation_stage(
    cand_out_dir, cand_stage_name, cand_manifest,
    feat_out_dir, feat_stage_name,
    s1_path, indexes, gt=None,
):
    """
    One feature shard per completed candidate shard (same shard index).
    gt=None (test): no labels, no sampling -- every candidate is scored later.
    gt given (train): validation-S1 rows always kept; train-S1 negatives
    are deterministically subsampled per TRAIN_NEGATIVE_KEEP_PROB.
    """
    feat_manifest_path = str(CKPT_DIR / f"manifest_{feat_stage_name}.json")
    feat_manifest = ckpt.load_manifest(
        feat_manifest_path, feat_stage_name,
        cand_manifest["shard_size"], cand_manifest["total_rows"],
    )

    attrs_stream = S1AttrsStream(s1_path)
    t0 = time.time()
    n_written = 0
    n_seen = 0

    for shard_idx in sorted(cand_manifest["completed_shards"]):

        if ckpt.is_shard_complete(feat_manifest, shard_idx):
            continue

        cand_path = ckpt.shard_output_path(cand_out_dir, f"cand_{cand_stage_name}", shard_idx)
        out_rows = []

        with open(cand_path, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                n_seen += 1
                s1_id = row["source1_entity_id"]
                cand_id = row["candidate_entity_id"]
                source = row["source"]

                label = None
                if gt is not None:
                    label = label_for_compact(s1_id, cand_id, gt)
                    if label == 0:
                        in_val = is_validation_s1(s1_id)
                        is_singleton_s1 = len(gt.get(s1_id, frozenset())) == 0
                        # Singleton (no-true-match) S1 entities in validation are ALWAYS
                        # fully kept, never downsampled -- CELL 18/19 explicitly need every
                        # singleton's full candidate set to evaluate whether the calibrated
                        # threshold correctly abstains on them (brief: "explicitly evaluate
                        # singleton handling").
                        if not (in_val and is_singleton_s1):
                            keep_prob = VALIDATION_NEGATIVE_KEEP_PROB if in_val else TRAIN_NEGATIVE_KEEP_PROB
                            if not deterministic_keep(s1_id + "|" + cand_id, keep_prob):
                                continue  # drop this negative -- never even normalized/featurized

                name1, address1, country1 = attrs_stream.get(s1_id)
                _, suffix = split_entity_id(cand_id)
                cand_attrs = indexes[source].get_attrs(suffix)
                name2, address2, country2 = cand_attrs if cand_attrs else ("", "", "")

                feats = calculate_features(name1, address1, country1, name2, address2, country2)
                out_row = {**row, **feats}
                if gt is not None:
                    out_row["label"] = label
                out_rows.append(out_row)

        columns = FEATURE_FILE_COLUMNS + (["label"] if gt is not None else [])
        feat_path = ckpt.shard_output_path(feat_out_dir, f"feat_{feat_stage_name}", shard_idx)

        def _write(tmp, _rows=out_rows, _cols=columns):
            with open(tmp, "w", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=_cols, delimiter="\t")
                w.writeheader()
                for r in _rows:
                    w.writerow(r)
        ckpt.write_shard_atomically(feat_path, _write)
        ckpt.mark_shard_complete(feat_manifest_path, feat_manifest, shard_idx)
        n_written += len(out_rows)

        if shard_idx % 10 == 0:
            print(f"[{feat_stage_name}] shard {shard_idx}: {len(out_rows):,} feature rows written "
                  f"(seen {n_seen:,} candidates so far, elapsed {time.time()-t0:.1f}s)")

    print(f"[{feat_stage_name}] done: {n_written:,} feature rows newly written "
          f"across {len(feat_manifest['completed_shards'])} shards, {time.time()-t0:.1f}s")
    return feat_manifest


train_feat_manifest = run_feature_generation_stage(
    str(SHARD_DIR), "train_candidates", train_cand_manifest,
    str(SHARD_DIR), "train_features",
    S1_TRAIN, train_indexes, gt=gt_train,
)

[sampling] TRAIN_NEGATIVE_KEEP_PROB      : 1.000000 (estimated population 376,650)
[sampling] VALIDATION_NEGATIVE_KEEP_PROB : 1.000000 (estimated population 94,162)
[sampling] positives are always kept in both splits.
[train_features] shard 0: 470,938 feature rows written (seen 470,938 candidates so far, elapsed 305.0s)
[train_features] done: 470,938 feature rows newly written across 1 shards, 305.0s


## CELL 16 — Train/validation split

Loads the sampled feature shards from CELL 15 and reports the resulting train/validation split. The split is **by S1 entity** (`is_validation_s1`, CELL 14), not by row, so every candidate pair for a given S1 stays entirely on one side -- otherwise the model would leak information about how many true matches an entity has across the split boundary.

In [25]:
def load_feature_shards(out_dir, stage_name, manifest, columns):
    frames = []
    for shard_idx in sorted(manifest["completed_shards"]):
        path = ckpt.shard_output_path(out_dir, f"feat_{stage_name}", shard_idx)
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path, sep="\t", dtype={"source1_entity_id": str, "candidate_entity_id": str})
        frames.append(df)
    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True)

train_feature_columns = FEATURE_FILE_COLUMNS + ["label"]
all_features_df = load_feature_shards(str(SHARD_DIR), "train_features", train_feat_manifest, train_feature_columns)

# Downcast to keep memory bounded on large runs.
float_cols = [c for c in FEATURE_COLUMNS if all_features_df[c].dtype == "float64"]
all_features_df[float_cols] = all_features_df[float_cols].astype("float32")

all_features_df["is_validation"] = all_features_df["source1_entity_id"].apply(is_validation_s1)

train_df = all_features_df[~all_features_df["is_validation"]].reset_index(drop=True)
val_df = all_features_df[all_features_df["is_validation"]].reset_index(drop=True)

n_train_s1 = train_df["source1_entity_id"].nunique()
n_val_s1 = val_df["source1_entity_id"].nunique()

print(f"[split] total feature rows       : {len(all_features_df):,}")
print(f"[split] train rows / S1 entities : {len(train_df):,} / {n_train_s1:,}")
print(f"[split] val rows   / S1 entities : {len(val_df):,} / {n_val_s1:,}")
print(f"[split] train positive rate      : {train_df['label'].mean():.4%}")
print(f"[split] val positive rate        : {val_df['label'].mean():.4%}")

n_val_singletons = (val_df.groupby("source1_entity_id")["label"].max() == 0).sum()
print(f"[split] validation singleton (no-true-match) S1 entities represented: {n_val_singletons:,}")

assert set(train_df["source1_entity_id"]).isdisjoint(set(val_df["source1_entity_id"])), \
    "train/validation split leaked an S1 entity across both sides"
print("\n[split] OK -- no S1 entity appears on both sides of the split.")

[split] total feature rows       : 470,938
[split] train rows / S1 entities : 387,805 / 1,622
[split] val rows   / S1 entities : 83,133 / 371
[split] train positive rate      : 0.0273%
[split] val positive rate        : 0.0241%
[split] validation singleton (no-true-match) S1 entities represented: 352

[split] OK -- no S1 entity appears on both sides of the split.


## CELL 17 — Model training

Binary match classifier. Positive = true S1-S2/S3 pair (from ground truth); negative = any other candidate pair. `HistGradientBoostingClassifier` (`class_weight="balanced"`) is the reproducible CPU default with no extra dependency; if `HAVE_LIGHTGBM` and `USE_GPU`, a GPU-capable LightGBM model is tried too and the better of the two (by validation macro F0.5 at threshold 0.5, before calibration) is kept -- the notebook still produces a correct model if GPU/LightGBM is unavailable.

In [26]:
from sklearn.metrics import fbeta_score, average_precision_score

MODEL_FEATURE_COLUMNS = FEATURE_COLUMNS + ["blocking_num_keys", "blocking_min_freq"]

X_train = train_df[MODEL_FEATURE_COLUMNS].fillna(0)
y_train = train_df["label"].astype(int)
X_val = val_df[MODEL_FEATURE_COLUMNS].fillna(0)
y_val = val_df["label"].astype(int)

from sklearn.ensemble import HistGradientBoostingClassifier

t0 = time.time()
hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.08,
    max_depth=8,
    class_weight="balanced",
    random_state=RANDOM_SEED,
)
hgb_model.fit(X_train, y_train)
print(f"[model] HistGradientBoostingClassifier trained in {time.time()-t0:.1f}s")

candidates_models = {"hgb": hgb_model}

if HAVE_LIGHTGBM and USE_GPU:
    try:
        import lightgbm as lgb
        t0 = time.time()
        pos = int(y_train.sum())
        neg = int(len(y_train) - pos)
        scale_pos_weight = (neg / pos) if pos else 1.0
        lgb_params = dict(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=63,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED,
        )
        try:
            lgb_model = lgb.LGBMClassifier(device="gpu", **lgb_params)
            lgb_model.fit(X_train, y_train)
        except Exception as gpu_err:
            print(f"[model] LightGBM GPU device unavailable ({gpu_err}); retrying on CPU.")
            lgb_model = lgb.LGBMClassifier(device="cpu", **lgb_params)
            lgb_model.fit(X_train, y_train)
        candidates_models["lgb"] = lgb_model
        print(f"[model] LightGBM trained in {time.time()-t0:.1f}s")
    except Exception as e:
        print(f"[model] LightGBM training failed, continuing with HistGradientBoostingClassifier only: {e}")

# Pick the model with the higher validation average precision (quick, threshold-free
# comparison) -- final decision threshold is calibrated properly in CELL 18.
best_name, best_ap = None, -1.0
for name, m in candidates_models.items():
    val_scores = m.predict_proba(X_val)[:, 1]
    ap = average_precision_score(y_val, val_scores)
    print(f"[model] {name}: validation average-precision = {ap:.4f}")
    if ap > best_ap:
        best_ap, best_name = ap, name

model = candidates_models[best_name]
print(f"\n[model] selected: {best_name} (validation AP = {best_ap:.4f})")

val_df = val_df.copy()
val_df["score"] = model.predict_proba(X_val)[:, 1]

[model] HistGradientBoostingClassifier trained in 3.3s
[LightGBM] [Info] Number of positive: 106, number of negative: 387699
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2809
[LightGBM] [Info] Number of data points in the train set: 387805, number of used features: 25
[model] LightGBM GPU device unavailable (No OpenCL device found); retrying on CPU.
[LightGBM] [Info] Number of positive: 106, number of negative: 387699
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030523 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2809
[LightGBM] [Info] Number of data points in the train set: 387805, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000273 -> initscore=-8.204545
[LightGBM] [Info] Start training from score -8.204545
[LightGBM] [Warning] No further splits with positive gain, bes

## CELL 18 — F0.5 threshold calibration

Macro-averaged F0.5: precision/recall/F0.5 computed **per S1 entity** on the validation set, then averaged across entities (not micro-averaged across all pairs) -- these differ whenever match counts per S1 vary, which they do here. Singleton entities (no true match) are included: a threshold calibrated only against entities that *have* a match would be systematically miscalibrated against them. Because F0.5 weights precision over recall, expect (and measure, not assume) a threshold above 0.5. A per-source (S2 vs S3) threshold is also measured and kept only if it clearly beats a single global threshold.

In [27]:
def macro_f05_by_threshold(df, threshold, beta=0.5):
    beta2 = beta ** 2
    f05_scores = []
    for _, g in df.groupby("source1_entity_id"):
        pred = g["score"] >= threshold
        tp = int((pred & (g["label"] == 1)).sum())
        fp = int((pred & (g["label"] == 0)).sum())
        fn = int((~pred & (g["label"] == 1)).sum())
        if tp + fp == 0:
            precision = 1.0 if fn == 0 else 0.0
        else:
            precision = tp / (tp + fp)
        recall = 1.0 if (tp + fn) == 0 else tp / (tp + fn)
        if precision + recall == 0:
            f05 = 0.0
        else:
            f05 = (1 + beta2) * precision * recall / (beta2 * precision + recall)
        f05_scores.append(f05)
    return float(np.mean(f05_scores)) if f05_scores else 0.0

threshold_grid = np.round(np.arange(0.05, 0.96, 0.01), 2)

t0 = time.time()
global_scores = [(t, macro_f05_by_threshold(val_df, t)) for t in threshold_grid]
best_global_threshold, best_global_f05 = max(global_scores, key=lambda x: x[1])
print(f"[calibration] global sweep done in {time.time()-t0:.1f}s")
print(f"[calibration] best global threshold: {best_global_threshold:.2f}  (macro F0.5 = {best_global_f05:.4f})")

# Per-source thresholds -- only adopted if they beat the global choice.
per_source_thresholds = {}
for source in val_df["source"].unique():
    sub = val_df[val_df["source"] == source]
    scores = [(t, macro_f05_by_threshold(sub, t)) for t in threshold_grid]
    best_t, best_f = max(scores, key=lambda x: x[1])
    per_source_thresholds[source] = {"threshold": float(best_t), "macro_f05": float(best_f)}
    print(f"[calibration] source={source}: best threshold={best_t:.2f}  macro F0.5={best_f:.4f}")

use_per_source = all(
    per_source_thresholds[s]["macro_f05"] >= best_global_f05 - 1e-9 for s in per_source_thresholds
) and len(per_source_thresholds) > 1

if use_per_source:
    combined_check = macro_f05_by_threshold(
        pd.concat([
            val_df[val_df["source"] == s].assign(score=lambda d, s=s: d["score"])
            for s in per_source_thresholds
        ]),
        best_global_threshold,
    )
    print("[calibration] per-source thresholds each match/beat the global threshold -- adopting per-source.")
    DECISION_THRESHOLDS = {s: per_source_thresholds[s]["threshold"] for s in per_source_thresholds}
else:
    print("[calibration] adopting a single global threshold.")
    DECISION_THRESHOLDS = {"__default__": float(best_global_threshold)}

print("\n[calibration] FINAL DECISION_THRESHOLDS:", DECISION_THRESHOLDS)

with open(CKPT_DIR / "decision_thresholds.json", "w") as f:
    json.dump(DECISION_THRESHOLDS, f, indent=2)


def decide(row):
    t = DECISION_THRESHOLDS.get(row["source"], DECISION_THRESHOLDS.get("__default__", 0.5))
    return row["score"] >= t

val_df["predicted"] = val_df.apply(decide, axis=1)

[calibration] global sweep done in 22.1s
[calibration] best global threshold: 0.94  (macro F0.5 = 0.9811)
[calibration] source=S2: best threshold=0.94  macro F0.5=0.9946
[calibration] source=S3: best threshold=0.64  macro F0.5=0.9864
[calibration] per-source thresholds each match/beat the global threshold -- adopting per-source.

[calibration] FINAL DECISION_THRESHOLDS: {'S2': 0.94, 'S3': 0.64}


## CELL 19 — Validation results

In [28]:
final_macro_f05 = macro_f05_by_threshold(val_df.assign(score=val_df["score"]), best_global_threshold) \
    if not use_per_source else np.mean([
        macro_f05_by_threshold(val_df[val_df["source"] == s], per_source_thresholds[s]["threshold"])
        for s in per_source_thresholds
    ])

tp = int(((val_df["predicted"]) & (val_df["label"] == 1)).sum())
fp = int(((val_df["predicted"]) & (val_df["label"] == 0)).sum())
fn = int(((~val_df["predicted"]) & (val_df["label"] == 1)).sum())
tn = int(((~val_df["predicted"]) & (val_df["label"] == 0)).sum())
micro_precision = tp / (tp + fp) if (tp + fp) else float("nan")
micro_recall = tp / (tp + fn) if (tp + fn) else float("nan")

# Singleton-specific check: of validation S1 entities with NO true match,
# how many get zero predicted matches (correct abstention)?
singleton_ids = val_df.groupby("source1_entity_id")["label"].max()
singleton_ids = singleton_ids[singleton_ids == 0].index
singleton_df = val_df[val_df["source1_entity_id"].isin(singleton_ids)]
singleton_any_fp = singleton_df.groupby("source1_entity_id")["predicted"].any()
singleton_correct = int((~singleton_any_fp).sum())
singleton_total = len(singleton_any_fp)

print("=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)
print(f"Macro F0.5 (calibrated threshold)   : {final_macro_f05:.4f}")
print(f"Micro precision                     : {micro_precision:.4f}")
print(f"Micro recall                        : {micro_recall:.4f}")
print(f"TP / FP / FN / TN                   : {tp:,} / {fp:,} / {fn:,} / {tn:,}")
print(f"Singleton S1 entities (no true match): {singleton_total:,}")
print(f"  correctly predicted 'no match'    : {singleton_correct:,} "
      f"({(singleton_correct/singleton_total if singleton_total else float('nan')):.2%})")

validation_report = {
    "macro_f05": final_macro_f05,
    "micro_precision": micro_precision,
    "micro_recall": micro_recall,
    "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    "decision_thresholds": DECISION_THRESHOLDS,
    "singleton_total": singleton_total,
    "singleton_correct": singleton_correct,
    "selected_model": best_name,
    "n_train_rows": len(train_df),
    "n_val_rows": len(val_df),
    "n_train_s1": n_train_s1,
    "n_val_s1": n_val_s1,
}
with open(OUT_DIR / "validation_report.json", "w") as f:
    json.dump(validation_report, f, indent=2)

import pickle
with open(CKPT_DIR / "final_model.pkl", "wb") as f:
    pickle.dump({"model": model, "feature_columns": MODEL_FEATURE_COLUMNS, "model_name": best_name}, f)

print("\n[saved]", OUT_DIR / "validation_report.json")
print("[saved]", CKPT_DIR / "final_model.pkl")

VALIDATION RESULTS
Macro F0.5 (calibrated threshold)   : 0.9905
Micro precision                     : 0.8421
Micro recall                        : 0.8000
TP / FP / FN / TN                   : 16 / 3 / 4 / 83,110
Singleton S1 entities (no true match): 352
  correctly predicted 'no match'    : 349 (99.15%)

[saved] /kaggle/working/amazon_ml_solution/output/validation_report.json
[saved] /kaggle/working/amazon_ml_solution/checkpoints/final_model.pkl
